# Market-behavior representation exploration

This exploratory representation summarizes each item in a Sunday–Saturday focal week, with immediately preceding and following week context. Notebook 01 establishes the study population; the formulas are exploratory descriptors, not economic conclusions. See [feature sources](feature_sources.md).

### Conventions

The applicable item-day calendar retains absent days as zero activity and missing prices, while observed nulls remain null. VWAP is the source average; historical order count is not trade count. Each focal and adjacent week needs at least five positive finite VWAP days. Returns use only adjacent valid calendar days, never a bridged gap. Eligible PCA rows require finite active features.

### Implementation choices and routine route

Run the workbench from setup through section 05. Section 06 provides optional retained-PCA diagnostics, item influence, and the temporal explorer, with one selectable view at a time. Controls are editable variables rather than widgets. Computational modules provide descriptor and joint-path preparation/fitting and robustness checks; plots remain notebook-local. Each fit is retained separately in memory with its input snapshots and execution-owned cache; all are lost on kernel restart. List retained executions and select one ID before rendering. These choices are provisional: they can be reconsidered if iteration, reuse, or persistence needs change. The five main figures are the two overviews and three run-inspection boards; section 05 boards are optional.

In [ ]:
from datetime import date

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
from plotly.subplots import make_subplots

from transform.notebook_access import open_analysis_connection

pio.renderers.default = "notebook"

HISTORY_START = date(2022, 6, 1)
HISTORY_END = date(2026, 9, 1)

In [ ]:
INPUT_SQL = """
select
    type_id, region_id, type_name, study_category, market_date,
    average as vwap, highest, lowest, volume, order_count, estimated_traded_isk
from main.int_study_panel_history
order by type_id, market_date
"""

with open_analysis_connection() as connection:
    panel_history = pl.DataFrame(connection.execute(INPUT_SQL).to_arrow_table())

panel_history = panel_history.filter(pl.col("market_date").is_between(HISTORY_START, HISTORY_END)).with_columns(
    pl.col("market_date").cast(pl.Date, strict=True)
)
if panel_history.is_empty():
    raise ValueError("The configured study period has no history.")
if panel_history.select(pl.any_horizontal(pl.col("type_id").is_null(), pl.col("market_date").is_null()).any()).item():
    raise ValueError("Expected nonmissing item IDs and dates.")
if panel_history.select(pl.struct("type_id", "market_date").is_duplicated().any()).item():
    raise ValueError("Expected one observed history row per item-date.")

## Weekly feature bank

Weeks run Sunday through Saturday. Price movement and excursions start at the last valid prior-week price and end at or include available focal prices; no returns are fabricated across gaps. Early/late medians split that available path into chronological non-overlapping halves. Timing uses actual weekday positions; residence, settling, alignment and prior/next context are defined in [feature sources](feature_sources.md). Coverage and activity totals are metadata, not model inputs.

In [ ]:
def build_applicable_history(history: pl.DataFrame) -> pl.DataFrame:
    """Materialize the established item-by-day calendar without filling observed values."""
    item_metadata = history.select("type_id", "region_id", "type_name", "study_category").unique(
        "type_id", maintain_order=True
    )
    calendar = pl.DataFrame(
        {"market_date": pl.date_range(HISTORY_START, HISTORY_END, "1d", eager=True)}
    )
    applicable = calendar.join(item_metadata, how="cross")
    source_values = history.drop("region_id", "type_name", "study_category").with_columns(
        pl.lit(True).alias("is_observed")
    )
    return (
        applicable.join(source_values, on=["type_id", "market_date"], how="left")
        .with_columns(pl.col("is_observed").fill_null(False))
        .with_columns(
            pl.when(~pl.col("is_observed")).then(pl.lit(0.0)).otherwise(pl.col("volume")).alias("volume"),
            pl.when(~pl.col("is_observed")).then(pl.lit(0.0)).otherwise(pl.col("order_count")).alias("order_count"),
        )
        .sort("type_id", "market_date")
    )



applicable_history = build_applicable_history(panel_history)

In [ ]:
# Edit this list to choose model coordinates; coverage fields remain metadata.
FEATURE_DESCRIPTORS = {
    "price_log_change": "Net price movement", "upward_excursion": "Upward excursion",
    "downward_excursion": "Downward excursion", "median_shift": "Early-to-late median shift",
    "mean_relative_range": "Mean relative range", "volume_ratio": "Focal/prior volume",
    "order_count_ratio": "Focal/prior order count", "next_volume_ratio": "Next/prior volume",
    "next_order_count_ratio": "Next/prior order count", "peak_day_volume_share": "Peak volume share",
    "directional_balance": "Directional balance", "peak_timing": "Peak timing",
    "trough_timing": "Trough timing", "upward_residence": "Upward residence",
    "downward_residence": "Downward residence", "terminal_settling": "Terminal settling",
    "price_activity_alignment": "Price–activity alignment", "focal_prior_price": "Focal/prior median price",
    "next_prior_price": "Next/prior median price",
}
FEATURE_COLUMNS = list(FEATURE_DESCRIPTORS)

def build_feature_bank(history: pl.DataFrame, *, origin_shift_days: int = 0) -> pl.DataFrame:
    """Build focal item-weeks with adjacent context; zero anchors on Sunday."""
    from datetime import timedelta

    if not isinstance(origin_shift_days, int) or isinstance(origin_shift_days, bool):
        raise ValueError("origin_shift_days must be an integer")
    weeks = {}
    for row in history.sort("type_id", "market_date").iter_rows(named=True):
        day = row["market_date"]
        start = day - timedelta(days=((day.weekday() + 1 - origin_shift_days) % 7))
        weeks.setdefault((row["type_id"], start), []).append(row)

    def positive(value):
        return value is not None and np.isfinite(value) and value > 0

    def nonnegative(value):
        return value is not None and np.isfinite(value) and value >= 0

    def activity(rows, name):
        vals = [r[name] for r in rows]
        return float(sum(vals)) if all(nonnegative(v) for v in vals) else None

    def ratio(numerator, denominator):
        return numerator / denominator if numerator is not None and denominator is not None and denominator > 0 else None

    records = []
    for (item, start), focal in weeks.items():
        prior = weeks.get((item, start - timedelta(days=7)), [])
        following = weeks.get((item, start + timedelta(days=7)), [])
        valid = lambda rows: [r for r in rows if positive(r["vwap"])]
        p, f, n = valid(prior), valid(focal), valid(following)
        if min(len(p), len(f), len(n)) < 5:
            continue
        boundary = p[-1]["vwap"]
        prices = np.array([r["vwap"] for r in f], dtype=float)
        path = np.r_[boundary, prices]
        split = len(path) // 2
        prior_median = float(np.median([r["vwap"] for r in p]))
        focal_median = float(np.median(prices))
        next_median = float(np.median([r["vwap"] for r in n]))
        log_displacement = np.log(prices / prior_median)
        up = np.maximum(log_displacement, 0)
        down = np.maximum(-log_displacement, 0)
        # Include the first focal day only when its preceding calendar day is usable;
        # the last valid prior price used for path boundaries must not bridge a gap.
        return_path = prior[-1:] + focal
        returns = [np.log(b["vwap"] / a["vwap"]) for a, b in zip(return_path, return_path[1:])
                   if b["market_date"] >= start and (b["market_date"] - a["market_date"]).days == 1
                   and positive(a["vwap"]) and positive(b["vwap"])]
        ranges = [(r["highest"] - r["lowest"]) / r["vwap"] for r in f
                  if nonnegative(r["lowest"]) and r["highest"] is not None
                  and np.isfinite(r["highest"]) and r["highest"] >= r["lowest"]]
        volumes = [activity(rows, "volume") for rows in (prior, focal, following)]
        orders = [activity(rows, "order_count") for rows in (prior, focal, following)]
        pairs = [(np.log(r["vwap"]), r["volume"]) for r in f if nonnegative(r["volume"])]
        paired_total = sum(v for _, v in pairs)
        alignment = (sum(x * v for x, v in pairs) / paired_total - np.mean([x for x, _ in pairs])) if paired_total > 0 else None
        weekdays = np.array([(r["market_date"] - start).days for r in f])
        record = dict(type_id=item, type_name=focal[0]["type_name"], study_category=focal[0]["study_category"],
                      week_start=start, week_end=start + timedelta(days=6),
                      prior_valid_price_days=len(p), valid_price_days=len(f), next_valid_price_days=len(n),
                      valid_return_pairs=len(returns), prior_volume_total=volumes[0], focal_volume_total=volumes[1], next_volume_total=volumes[2],
                      prior_order_count_total=orders[0], focal_order_count_total=orders[1], next_order_count_total=orders[2],
                      price_log_change=np.log(prices[-1] / boundary),
                      upward_excursion=np.log(max(boundary, float(prices.max())) / boundary),
                      downward_excursion=np.log(boundary / min(boundary, float(prices.min()))),
                      median_shift=np.log(np.median(path[split:]) / np.median(path[:split])),
                      mean_relative_range=float(np.mean(ranges)) if ranges else None,
                      volume_ratio=ratio(volumes[1], volumes[0]), order_count_ratio=ratio(orders[1], orders[0]),
                      next_volume_ratio=ratio(volumes[2], volumes[0]), next_order_count_ratio=ratio(orders[2], orders[0]),
                      peak_day_volume_share=max(r["volume"] for r in focal) / volumes[1] if volumes[1] is not None and volumes[1] > 0 else None,
                      directional_balance=float(np.mean(np.sign(returns))) if returns else None,
                      peak_timing=float(np.mean(weekdays[prices == prices.max()]) / 6),
                      trough_timing=float(np.mean(weekdays[prices == prices.min()]) / 6),
                      upward_residence=float(np.mean(up) / up.max()) if up.max() > 0 else 0.0,
                      downward_residence=float(np.mean(down) / down.max()) if down.max() > 0 else 0.0,
                      terminal_settling=float(np.mean(np.abs(returns[-3:]))) if returns else None,
                      price_activity_alignment=alignment,
                      focal_prior_price=np.log(focal_median / prior_median), next_prior_price=np.log(next_median / prior_median))
        records.append(record)
    return pl.DataFrame(records)

descriptor_bank_history = applicable_history.clone()
behavior_rows = build_feature_bank(descriptor_bank_history)
descriptor_bank_snapshot = behavior_rows.clone()
assert behavior_rows.select("type_id", "week_start").n_unique() == behavior_rows.height

## 00 — Question and batch controls

Which retrospective weekly behaviors remain interpretable across descriptor representations and cluster counts? Each row includes prior and following-week context, so these are not predictive features. The default uses all 19 existing weekly features; edit `FEATURE_COLUMNS` in the feature-bank cell to compare a different set, then rebuild the bank and batch. Finiteness and the original three-week price-support rule determine eligibility.

### Representation and fitting choices

The browsing package compares the unreduced prepared input with PCA dimensions 3, 6, 10 and UMAP dimensions 3, 6 with neighbors 15, 50, 150. It uses K = 3, 4, 6, 8, 12, one repetition and five K-means initializations. Autoencoder and DTW are opt-in. The editable budgets are advisory, not limits; invalid dimensions and other skips are reported before fitting. AE retains its existing training defaults, calendar purge and full-data refit.

In [ ]:
from pathlib import Path
import sys
from dataclasses import replace

# Support execution from the repository, experiments, or notebook directory.
_workbench_cwd = Path.cwd().resolve()
_workbench_roots = [p / "experiments" for p in (_workbench_cwd, *_workbench_cwd.parents)] + [_workbench_cwd, *_workbench_cwd.parents]
_experiments_root = next((p for p in _workbench_roots if (p / "market_analysis" / "behavior_workbench.py").is_file()), None)
if _experiments_root is None:
    raise RuntimeError("Cannot locate experiments/market_analysis from the current directory")
if str(_experiments_root) not in sys.path:
    sys.path.insert(0, str(_experiments_root))
from market_analysis.behavior_workbench import ExecutionRegistry, Settings, plan, prepare, explore, path_plan, explore_paths
from market_analysis.behavior_paths import prepare_joint_paths

batch_settings = Settings(
    # Default PCA grid (3, 6, 10) omits full-width PCA when it duplicates prepared.
    pca_dimensions=None, umap_dimensions=(3, 6), ae_dimensions=(3, 6),
    clusters=(3, 4, 6, 8, 12), neighbors=(15, 50, 150),
    methods=("prepared", "pca", "umap", "ae"), ae_enabled=False, ae_weight_decays=(1e-4,),
    learned_representation_budget=10, clustering_budget=60,  # advisory, never truncates
    seed=42, repetitions=1, bound_ratios=True, strength=1.5,
    umap_min_dist=0.1, kmeans_n_init=5, ae_max_epochs=100, ae_patience=10,
)

## 01 — Prepare descriptor input

The adapter preserves item–week keys, coverage metadata and the bank's raw feature values. Standardization, magnitude compression and bounded activity ratios retain the existing geometry without a second rescaling. Change the ratio toggle or strength only as an explicit geometry comparison; changes require a new fit. Estimated array sizes exclude models and temporary allocations. Actual fit elapsed time and, where available, process-lifetime peak RSS are reported after each explicit fit.

In [ ]:
import hashlib

def source_fingerprint(*frames):
    """In-process provenance for ordered, typed source snapshots."""
    digest = hashlib.sha256()
    for frame in frames:
        digest.update(str(frame.schema).encode())
        digest.update(str(frame.shape).encode())
        digest.update(frame.hash_rows(seed=0).to_numpy().tobytes())
    return digest.hexdigest()

# Freeze the feature bank together with the exact history from which it was built.
_descriptor_bank = descriptor_bank_snapshot.clone()
_descriptor_history = descriptor_bank_history.clone()
descriptor_features = tuple(FEATURE_COLUMNS)
descriptor_keys = tuple(_descriptor_bank.select("type_id", "week_start").iter_rows())
descriptor_weeks = _descriptor_bank["week_start"].to_numpy().copy()
descriptor_raw = _descriptor_bank.select(pl.col(c).cast(pl.Float64) for c in descriptor_features).to_numpy().copy()
descriptor_raw.setflags(write=False)
descriptor_source_fingerprint = source_fingerprint(_descriptor_bank, _descriptor_history)
descriptor_metadata = {"family": "descriptor", "key_columns": ("type_id", "week_start"),
                       "support": "prior, focal and following weeks; at least five valid prices each",
                       "source_fingerprint": descriptor_source_fingerprint}
descriptor_prepared_input = {"bank": _descriptor_bank, "source_history": _descriptor_history,
    "keys": descriptor_keys, "features": descriptor_features, "raw": descriptor_raw,
    "source_fingerprint": descriptor_source_fingerprint}
batch_estimate = plan(descriptor_raw, batch_settings)

In [ ]:
print(f"Eligible weeks: {batch_estimate['eligible_rows']:,} / {batch_estimate['input_rows']:,}; "
      f"representation fits: {batch_estimate['representation_fits']} "
      f"({batch_estimate['representation_skipped']} skipped: disabled, invalid or duplicate); "
      f"clustering fits: {batch_estimate['clustering_fits']}; "
      f"repetitions: {batch_settings.repetitions}.")
print(f"Prepared matrix + embeddings: approximately {(batch_estimate['prepared_bytes'] + batch_estimate['embedding_bytes_approx']) / 2**20:.1f} MiB; estimated array sizes; excludes models and temporary allocations.")

### Enabled families and phase seed

Both descriptor and joint-path fitting are enabled by default; remove either name to fit the other independently. Run the phase-configuration cell once for a new fitting phase. Fixed seed 42 is reproducible; with `batch_settings.seed = None` it draws and retains one random master seed for all enabled families and path subsets. Display controls do not resolve seeds or fit anything. Existing retained path batches are not silently selected or refitted.

In [ ]:
enabled_families = ("descriptor", "joint_paths")  # choose either or both
descriptor_label = None  # optional descriptive label
if not set(enabled_families) <= {"descriptor", "joint_paths"}:
    raise ValueError("Unknown input family")
import secrets
resolved_master_seed = batch_settings.seed if batch_settings.seed is not None else secrets.randbits(32)
if "execution_registry" not in globals():
    execution_registry = ExecutionRegistry()

def report_batch_progress(record):
    if "representation_id" not in record:
        p = record["params"]
        detail = ", ".join(f"{key}={p[key]}" for key in ("dimension", "neighbors", "weight_decay", "band", "repetition") if key in p)
        print(f"{p['method']} {detail}: {record['status']}")
    elif record["status"] == "optimization-incomplete":
        p, details = record["params"], record.get("details", {})
        print(f"{p['method']} K={record['k']} repetition={p.get('repetition', 0)}: "
              f"{record['status']} · {record.get('reason')} · "
              f"converged={details.get('converged')} iterations={details.get('iterations')} "
              f"objective={details.get('objective')} · run={record['id']}")

print(f"Resolved master seed for this fitting phase: {resolved_master_seed}")

def require_active_execution():
    """Reject missing, failed, or out-of-order selections at every render."""
    if globals().get("active_execution_id") is None or execution_registry.active_id != active_execution_id:
        raise ValueError("Select a retained execution using the active execution cell")
    execution = execution_registry.active
    if execution is None or execution.options.get("family") not in ("descriptor", "joint_paths"):
        raise ValueError("Invalid active execution")
    if execution.batch.input_metadata.get("family") != execution.options["family"]:
        raise ValueError("Active execution family does not match its batch")
    return execution

def active_batch_args():
    execution = require_active_execution()
    return execution.batch, execution.options["family"]

def activate_first_successful_execution():
    """Keep an active execution; otherwise select the first successful batch."""
    global active_execution_id
    if execution_registry.active_id is None:
        first = next((e for e in execution_registry.list()
                      if any(r.get("status") == "ok" for r in e.batch.runs.values())), None)
        if first is not None:
            execution_registry.select(first.id)
            print(f"Automatically activated execution: {first.id}")
    active_execution_id = execution_registry.active_id
    return active_execution_id

## 02 — Fit exploration batch

Define the descriptor fit here, prepare joint paths and review the combined plan below, then run the final fit cell only when inputs or fit controls change. Each representation is fitted once per configuration and repetition and reused across K values. Results, keys, diagnostics and failure reasons stay in memory; rerendering either overview does not fit or generate seeds.

In [ ]:
def report_batch_progress(record):
    if "representation_id" not in record:
        p = record["params"]
        detail = ", ".join(f"{key}={p[key]}" for key in ("dimension", "neighbors", "weight_decay", "band", "repetition") if key in p)
        print(f"{p['method']} {detail}: {record['status']}")
    elif record["status"] == "optimization-incomplete":
        p, details = record["params"], record.get("details", {})
        print(f"{p['method']} K={record['k']} repetition={p.get('repetition', 0)}: "
              f"{record['status']} · {record.get('reason')} · "
              f"converged={details.get('converged')} iterations={details.get('iterations')} "
              f"objective={details.get('objective')} · run={record['id']}")

def measure_explicit_fit(label, fit):
    """Time one fit call, including retention and activation, not prior preparation."""
    import sys
    from time import perf_counter
    started = perf_counter()
    try:
        return fit()
    finally:
        elapsed = perf_counter() - started
        peak = "process-lifetime peak RSS unavailable"
        try:
            import resource
            rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            if sys.platform.startswith("linux"):
                peak = f"process-lifetime peak RSS {rss / 1024:.1f} MiB"
            elif sys.platform == "darwin":
                peak = f"process-lifetime peak RSS {rss / 2**20:.1f} MiB"
        except (ImportError, OSError, ValueError):
            pass
        print(f"{label} fit call elapsed {elapsed:.3f} s (including retention/activation; "
              f"excluding prior input preparation); {peak} (not per-fit allocation).")

def fit_descriptor_execution():
    descriptor_settings = replace(batch_settings, seed=resolved_master_seed)
    batch = explore(descriptor_keys, descriptor_raw, descriptor_features, descriptor_weeks,
                             descriptor_settings, input_metadata=descriptor_metadata, progress=report_batch_progress)
    execution_id = execution_registry.retain(
        batch, descriptor_settings, preparation=descriptor_prepared_input, label=descriptor_label,
        source={"bank": _descriptor_bank, "history": _descriptor_history},
        options={"family": "descriptor", "input_metadata": descriptor_metadata},
    ).id
    print(f"Actual master seed: {batch.master_seed}; population: {batch.population_id[:12]}; execution: {execution_id}")
    activate_first_successful_execution()

# Both enabled families are fitted in cell 22, after the combined plan is reported.

### Joint-path preparation and fit

The 14-day windows use a seven-day stride and a preceding 14-day reference with at least 10 observed positive prices. Volume is the default activity channel; choose order count for an alternate view. Upstream absent days have zero activity, while observed null activity is ineligible. Prices use log relative to the reference median and activity uses log1p relative to its reference mean. Pooled channel standard deviations give equal-scale channels without per-episode normalization. Candidate rejection reasons, source-observation and interpolation flags, original daily observations, selected keys, and the full source-history snapshot are retained for later inspection.

Aligned Euclidean fitting is enabled. DTW is opt-in; enabling it prepares one deterministic matched subset (suggested 2,000, configurable without enforcement), and both methods use exactly that subset and its pooled scales. A separately retained full Euclidean population is distinct. The final fit cell fits all enabled families; to rerun only paths, disable the descriptor family for that phase.

In [ ]:
joint_activity = "volume"  # or "order_count"
joint_window = 14
joint_stride = 7
joint_reference_days = 14
joint_min_price_reference = 10
joint_include_dtw = False
joint_subset_size = 2000  # used only when DTW is enabled; configurable suggestion
joint_dtw_band = 2
joint_medoid_max_iter = 100
joint_batch_label = "matched_dtw" if joint_include_dtw else "full_euclidean"  # optional descriptive label

In [ ]:
# Execute after resolving the phase seed; this cell does not fit descriptors.
if "joint_paths" in enabled_families:
    if joint_activity not in ("volume", "order_count"):
        raise ValueError("Select volume or order_count")
    joint_source_history = applicable_history.clone()
    joint_source_fingerprint = source_fingerprint(joint_source_history)
    joint_control_metadata = {
        "source_fingerprint": joint_source_fingerprint,
        "window": joint_window, "stride": joint_stride, "reference_days": joint_reference_days,
        "min_price_reference": joint_min_price_reference, "activity_column": joint_activity,
        "price_column": "vwap", "date_column": "market_date", "observed_column": "is_observed",
        "market_columns": ("region_id", "type_id"), "subset_size": joint_subset_size if joint_include_dtw else None,
        "subset_seed": resolved_master_seed if joint_include_dtw else None,
    }
    joint_fit_options = {"include_dtw": joint_include_dtw, "dtw_band": joint_dtw_band,
        "medoid_max_iter": joint_medoid_max_iter}
    joint_history = joint_source_history.select("region_id", "type_id", "market_date", "vwap", joint_activity, "is_observed").to_pandas()
    joint_paths = prepare_joint_paths(joint_history, market_columns=("region_id", "type_id"),
        date_column="market_date", price_column="vwap", activity_column=joint_activity,
        observed_column="is_observed", window=joint_window, stride=joint_stride,
        reference_days=joint_reference_days, min_price_reference=joint_min_price_reference,
        subset_size=joint_subset_size if joint_include_dtw else None, seed=resolved_master_seed)
    joint_market_metadata = joint_source_history.select("region_id", "type_id", "type_name", "study_category").unique().to_pandas()
    if joint_market_metadata.duplicated(["region_id", "type_id"]).any():
        raise ValueError("Market metadata must be unique")
    joint_candidate_metadata = joint_paths.candidates.merge(joint_market_metadata, on=["region_id", "type_id"], how="left", validate="many_to_one")
    joint_selected_metadata = joint_paths.keys.merge(joint_market_metadata, on=["region_id", "type_id"], how="left", validate="many_to_one")
    joint_settings = replace(batch_settings, seed=resolved_master_seed)
    joint_estimate = path_plan(joint_paths, joint_settings, include_dtw=joint_include_dtw)
else:
    print("Joint-path preparation disabled for this phase.")

In [ ]:
# Combined planned attempts for enabled families; advisory budgets do not limit work.
descriptor_plan = batch_estimate if "descriptor" in enabled_families else None
joint_plan = joint_estimate if "joint_paths" in enabled_families else None
learned = descriptor_plan["learned_representation_fits"] if descriptor_plan else 0
cluster_attempts = sum(p["clustering_fits"] for p in (descriptor_plan, joint_plan) if p is not None)
print(f"Enabled families: {enabled_families}; learned representations: {learned} / advisory {batch_settings.learned_representation_budget}; "
      f"clustering fits: {cluster_attempts} / advisory {batch_settings.clustering_budget}.")
if learned > batch_settings.learned_representation_budget or cluster_attempts > batch_settings.clustering_budget:
    print("Advisory budget exceeded; no work is truncated. Edit settings if desired before fitting.")
if descriptor_plan:
    print(f"Descriptors: {descriptor_plan['eligible_rows']:,} eligible; {descriptor_plan['unreduced_spaces']} unreduced spaces; "
          f"{descriptor_plan['representation_fits']} learned fits; {descriptor_plan['clustering_fits']} cluster fits; "
          f"{descriptor_plan['representation_skipped']} representation and {descriptor_plan['clustering_skipped']} cluster configurations skipped "
          f"(invalid dimensions: {descriptor_plan['invalid_dimension_skipped']}; duplicate full PCA: {descriptor_plan['duplicate_full_pca_skipped']}; remainder disabled).")
    print(f"Descriptor prepared matrix + embeddings: approximately {(descriptor_plan['prepared_bytes'] + descriptor_plan['embedding_bytes_approx']) / 2**20:.1f} MiB.")
if joint_plan:
    print(f"Joint candidates: {len(joint_paths.candidates):,}; eligible: {int(joint_paths.candidates.eligible.sum()):,}; selected: {len(joint_paths.keys):,}; "
          f"{joint_plan['representation_fits']} preparations; {joint_plan['clustering_fits']} cluster fits; "
          f"{joint_plan['representation_skipped']} disabled preparation(s), {joint_plan['clustering_skipped']} disabled cluster configurations.")
    print(f"Joint tensor: {joint_plan['prepared_bytes'] / 2**20:.1f} MiB; DTW pairwise estimate: {joint_plan['pairwise_bytes'] / 2**30:.2f} GiB.")
print("Memory estimates exclude models and temporary allocations. Counts and budgets are advisory and do not establish affordability.")

In [ ]:
def fit_joint_execution():
    batch = explore_paths(joint_paths, joint_settings, **joint_fit_options,
        input_metadata=joint_control_metadata, progress=report_batch_progress)
    execution_id = execution_registry.retain(
        batch, joint_settings, label=joint_batch_label,
        preparation={"paths": joint_paths, "candidates": joint_candidate_metadata,
            "selected": joint_selected_metadata, "controls": joint_control_metadata,
            "source_fingerprint": joint_source_fingerprint, "population_id": batch.population_id},
        source={"history": joint_source_history},
        options={"family": "joint_paths", **joint_control_metadata, **joint_fit_options},
    ).id
    print(f"Actual master seed: {batch.master_seed}; population: {batch.population_id[:12]}; execution: {execution_id}; label: {joint_batch_label}")
    activate_first_successful_execution()

if "descriptor" in enabled_families:
    measure_explicit_fit("descriptor", fit_descriptor_execution)
else:
    print("Descriptor fitting disabled for this phase.")

if "joint_paths" in enabled_families:
    measure_explicit_fit("joint_paths", fit_joint_execution)
else:
    print("Joint-path fitting disabled for this phase.")

## Retained executions

List a page, then copy an execution ID into the selector. A fit preserves the active selection, or activates the first successful execution when none is active. Retained results and caches exist only for the lifetime of the kernel.

In [ ]:
execution_page = 0  # zero-based; 20 executions per page
if not isinstance(execution_page, int) or execution_page < 0:
    raise ValueError("execution_page must be a non-negative integer")
pl.DataFrame([
    {"id": execution.id, "time": execution.created_at.isoformat(),
     "label": execution.label, "family": execution.options["family"],
     "population_count": len(execution.batch.keys)}
    for execution in execution_registry.list()[execution_page * 20:(execution_page + 1) * 20]
], schema={"id": pl.String, "time": pl.String, "label": pl.String,
           "family": pl.String, "population_count": pl.Int64})

### Active execution

The current selection is retained by default. Paste an ID from the listing, or explicitly set `None` to clear it.

In [ ]:
active_execution_id = execution_registry.active_id  # paste a full ID, or None to clear
execution_registry.select(active_execution_id)
# The helper is defined later in the inspection controls on the first pass.
inspection_state = (inspection_state_for(require_active_execution())
                    if active_execution_id is not None and "inspection_state_for" in globals() else None)
print(f"Active execution: {execution_registry.active_id}")

### Retained executions → Cleanup

List full IDs explicitly and arm cleanup only when ready. The active execution cannot be deleted.

In [ ]:
cleanup_execution_ids = ()  # full execution IDs only
cleanup_armed = False
if cleanup_armed:
    if not cleanup_execution_ids:
        raise ValueError("Specify at least one execution ID for cleanup")
    execution_registry.delete(cleanup_execution_ids)
    print(f"Deleted {len(set(cleanup_execution_ids))} execution(s).")
else:
    print("Cleanup is not armed.")

## 03 — Compare runs

### Search overview controls

The single active execution determines the family and population for this overview. Only successful runs with defined silhouettes appear. Common-input separation is the default; native scores across geometries are not directly equivalent. Individual seeds are shown by default; summary averages repetitions within the same configuration. Largest-group fraction indicates imbalance. Controls belong to the active execution: edit the examples in the next cell to change them; switching executions restores each execution’s choices. Filtering and rerendering never fit.

In [ ]:
def comparison_controls(execution):
    """Editable comparison controls belong to the retained execution."""
    return execution.cache.setdefault("comparison_controls", {
        "overview_methods": ("prepared", "pca", "umap", "ae", "aligned", "dtw"),
        "overview_diagnostic": "common_input", "overview_seed_view": "individual",
        "landscape_method": "prepared", "landscape_diagnostic": "common_input",
        "landscape_repetition": 0, "landscape_neighbors": (15, 50, 150),
        "landscape_weight_decays": (1e-4,),
    })

def comparison_state(execution):
    """Repair controls against this execution's representations and runs."""
    state = comparison_controls(execution)
    reps = [rep["params"] for rep in execution.batch.representations.values()]
    preference = ("prepared", "pca", "umap", "ae") if execution.options["family"] == "descriptor" else ("aligned", "dtw")
    available = [m for m in preference if any(p["method"] == m for p in reps)]
    changes = []
    def repair(name, value):
        if state[name] != value:
            state[name] = value
            changes.append(f"{name} → {value}")
    if available:
        filtered = tuple(m for m in state["overview_methods"] if m in available)
        repair("overview_methods", filtered or tuple(available))
        if state["landscape_method"] not in available:
            repair("landscape_method", available[0])
        pairs = [(run, execution.batch.representations[run["representation_id"]]["params"])
                 for run in execution.batch.runs.values()]
        pairs = [(run, p) for run, p in pairs if p["method"] == state["landscape_method"]]
        repetitions = sorted({p.get("repetition", run["params"].get("repetition", 0)) for run, p in pairs})
        if repetitions and state["landscape_repetition"] not in repetitions:
            repair("landscape_repetition", repetitions[0])
        selected = [p for run, p in pairs if p.get("repetition", run["params"].get("repetition", 0)) == state["landscape_repetition"]]
        for method, key, name in (("umap", "neighbors", "landscape_neighbors"), ("ae", "weight_decay", "landscape_weight_decays")):
            if state["landscape_method"] == method:
                options = sorted({p[key] for p in selected if key in p})
                valid = tuple(dict.fromkeys(v for v in state[name] if v in options))[:6]
                if options:
                    repair(name, valid or tuple(options[:6]))
    if changes:
        print("Comparison controls reset: " + "; ".join(changes))
    return state

def overview_args():
    execution = require_active_execution()
    state = comparison_state(execution)
    return (execution.batch, execution.options["family"], state["overview_methods"],
            state["overview_diagnostic"], state["overview_seed_view"])

# Edit the active execution's controls as needed, then rerun the figure cell:
# comparison_controls(require_active_execution())["overview_methods"] = ("pca", "umap")
# comparison_controls(require_active_execution())["overview_diagnostic"] = "native"
# comparison_controls(require_active_execution())["overview_seed_view"] = "summary"

In [ ]:
def attempt_status_summary(runs):
    from collections import Counter
    counts = Counter(run["status"] for run in runs)
    return "Attempts: " + (" · ".join(f"{status}={count}" for status, count in sorted(counts.items())) if counts else "none")

def plot_search_overview(batch, family, methods, diagnostic, seed_view):
    import pandas as pd
    if family not in ("descriptor", "joint_paths") or batch.input_metadata.get("family") != family:
        raise ValueError("The selected batch and family must match")
    if diagnostic not in ("native", "common_input") or seed_view not in ("individual", "summary"):
        raise ValueError("Invalid diagnostic or seed view")
    rows = []
    attempts = [run for run in batch.runs.values()
                if batch.representations[run["representation_id"]]["params"]["method"] in methods]
    for run in attempts:
        rep = batch.representations[run["representation_id"]]
        p = rep["params"]
        if p["method"] not in methods or run["status"] != "ok":
            continue
        d = run[diagnostic]
        if d["silhouette"] is None:
            continue
        # Exclude only repetition from the grouping key, preserving all actual settings.
        config = {k: v for k, v in p.items() if k != "repetition"}
        cluster_config = {k: v for k, v in run["params"].items() if k not in ("repetition", "n_clusters")}
        rows.append(dict(method=p["method"], config=repr(config), cluster_config=repr(cluster_config),
                         k=run["k"], score=d["silhouette"], largest=d["largest_fraction"],
                         run_id=run["id"], seed=run["seed"], rep_seed=rep["seed"], repetition=p.get("repetition", run["params"].get("repetition", 0))))
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Separation vs K", "Imbalance vs separation"))
    if rows:
        frame = pd.DataFrame(rows)
        if seed_view == "summary":
            frame = frame.groupby(["method", "config", "cluster_config", "k"], as_index=False).agg(
                score=("score", "mean"), largest=("largest", "mean"), run_id=("run_id", lambda x: ", ".join(x)),
                seed=("seed", lambda x: ", ".join(map(str, x))), rep_seed=("rep_seed", lambda x: ", ".join(map(str, x))),
                repetition=("repetition", "count"))
        colors = {"pca": "#1f77b4", "umap": "#ff7f0e", "ae": "#2ca02c", "aligned": "#9467bd", "dtw": "#d62728"}
        for method, part in frame.groupby("method", sort=False):
            hover = [f"run(s): {r.run_id}<br>cluster seed(s): {r.seed}<br>representation seed(s): {r.rep_seed}<br>config: {r.config}<br>clustering: {r.cluster_config}<br>repetition/count: {r.repetition}" for r in part.itertuples()]
            fig.add_trace(go.Scatter(x=part.k, y=part.score, mode="markers", name=method, marker_color=colors.get(method), text=hover, hovertemplate="%{text}<extra></extra>"), row=1, col=1)
            fig.add_trace(go.Scatter(x=part.score, y=part.largest, mode="markers", name=method, showlegend=False, marker_color=colors.get(method), text=hover, hovertemplate="%{text}<extra></extra>"), row=1, col=2)
    else:
        fig.add_annotation(text="No successful runs with a defined silhouette for these filters", x=.5, y=.5, xref="paper", yref="paper", showarrow=False)
    fig.update_xaxes(title_text="K", row=1, col=1)
    fig.update_yaxes(title_text=f"{diagnostic} silhouette", row=1, col=1)
    fig.update_xaxes(title_text=f"{diagnostic} silhouette", row=1, col=2)
    fig.update_yaxes(title_text="Largest group fraction", row=1, col=2)
    fig.update_layout(title=f"{family} population {batch.population_id[:12]} · {seed_view}<br><sup>{attempt_status_summary(attempts)}</sup>")
    return fig

In [ ]:
plot_search_overview(*overview_args())

### Hyperparameter landscape controls

Choose a method, diagnostic and repetition for the active execution. Descriptor views show dimension versus K, faceted by selected UMAP neighborhoods or AE weight decays (at most six); prepared input and PCA have one facet. Edit the examples in the control cell to change the active execution’s stored choices. Compatible choices persist across execution switches; incompatible methods, repetitions and facets reset visibly. The default AE decay selector is 1e-4, matching the default fit grid; select other fitted decays to compare regularization, including disabled/skipped configurations without training. Joint-path views show K against the actual activity channel and alignment/band, not an invented embedding dimension. The selected batch fixes all other path controls and population. Failed, skipped, uncomputed and undefined silhouettes are labeled separately from low scores.

In [ ]:
def landscape_args():
    execution = require_active_execution()
    state = comparison_state(execution)
    return (execution.batch, execution.options["family"], state["landscape_method"],
            state["landscape_diagnostic"], state["landscape_repetition"],
            state["landscape_neighbors"], state["landscape_weight_decays"])

# Edit the active execution's controls as needed, then rerun the figure cell:
# comparison_controls(require_active_execution())["landscape_method"] = "umap"  # prepared/pca/umap/ae or aligned/dtw
# comparison_controls(require_active_execution())["landscape_diagnostic"] = "native"
# comparison_controls(require_active_execution())["landscape_repetition"] = 0
# comparison_controls(require_active_execution())["landscape_neighbors"] = (15, 50, 150)
# comparison_controls(require_active_execution())["landscape_weight_decays"] = (1e-4,)

In [ ]:
def plot_hyperparameter_landscape(batch, family, method, diagnostic, repetition, neighbors, weight_decays):
    if family not in ("descriptor", "joint_paths") or batch.input_metadata.get("family") != family:
        raise ValueError("The selected batch and family must match")
    allowed = ("prepared", "pca", "umap", "ae") if family == "descriptor" else ("aligned", "dtw")
    if method not in allowed or diagnostic not in ("native", "common_input"):
        raise ValueError("Invalid method or diagnostic for this family")
    facets = (tuple(neighbors) if family == "descriptor" and method == "umap" else
              tuple(weight_decays) if family == "descriptor" and method == "ae" else (None,))
    if not facets or len(facets) > 6 or len(set(facets)) != len(facets):
        raise ValueError("Select one to six distinct facets")
    matching = [(r, batch.representations[r["representation_id"]]) for r in batch.runs.values()]
    matching = [(r, rep) for r, rep in matching if rep["params"]["method"] == method and rep["params"].get("repetition", r["params"].get("repetition", 0)) == repetition]
    matching = [(r, rep) for r, rep in matching
                if (method != "umap" or rep["params"].get("neighbors") in facets)
                and (method != "ae" or rep["params"].get("weight_decay") in facets)]
    ks = sorted({r["k"] for r, _ in matching})
    if family == "descriptor":
        ys = sorted({rep["params"]["dimension"] for _, rep in matching})
    else:
        activity = batch.input_metadata["channels"][1]
        ys = sorted({f"{activity} · " + (f"DTW band={rep['params']['band']}" if method == "dtw" else "aligned Euclidean") for _, rep in matching})
    titles = [f"neighbors={v}" if method == "umap" else f"weight decay={v:g}" if method == "ae" else method for v in facets]
    fig = make_subplots(rows=1, cols=len(facets), subplot_titles=titles)
    for i, facet in enumerate(facets, 1):
        z = np.full((len(ys), len(ks)), np.nan)
        text = np.full((len(ys), len(ks)), "uncomputed", dtype=object)
        for r, rep in matching:
            p = rep["params"]
            if family == "descriptor" and method == "umap" and p.get("neighbors") != facet:
                continue
            if family == "descriptor" and method == "ae" and p.get("weight_decay") != facet:
                continue
            label = p["dimension"] if family == "descriptor" else f"{activity} · " + (f"DTW band={p['band']}" if method == "dtw" else "aligned Euclidean")
            y, x = ys.index(label), ks.index(r["k"])
            score = r.get(diagnostic, {}).get("silhouette")
            config = f"representation: {p}<br>clustering: {r['params']}"
            identity = f"run: {r['id']}<br>cluster seed: {r['seed']}<br>representation seed: {rep['seed']}<br>{config}"
            if r["status"] == "ok" and score is not None:
                z[y, x] = score
                text[y, x] = f"{score:.3f}<br>{identity}"
            else:
                status = r["status"] if r["status"] != "ok" else "undefined"
                reason = r.get("reason") or r.get(diagnostic, {}).get("silhouette_reason") or "silhouette unavailable"
                text[y, x] = f"{status}<br>{reason}<br>{identity}"
        fig.add_trace(go.Heatmap(z=z, x=ks, y=ys, coloraxis="coloraxis", text=text, hovertemplate="K=%{x}<br>%{y}<br>%{text}<extra></extra>"), row=1, col=i)
        for y, label in enumerate(ys):
            for x, k in enumerate(ks):
                if np.isnan(z[y, x]):
                    fig.add_annotation(x=k, y=label, text=str(text[y, x]).split("<br>")[0], hovertext=str(text[y, x]), showarrow=False, row=1, col=i)
        fig.update_xaxes(title_text="K", row=1, col=i)
        fig.update_yaxes(title_text="Dimension" if family == "descriptor" else "Activity / alignment", row=1, col=i)
    if not ys or not ks:
        fig.add_annotation(text="No configurations for this method/repetition", x=.5, y=.5, xref="paper", yref="paper", showarrow=False)
    fig.update_layout(title=f"{family} · {method} · {diagnostic} · repetition {repetition} · population {batch.population_id[:12]}<br><sup>{attempt_status_summary(r for r, _ in matching)}</sup>", coloraxis=dict(colorscale="Viridis", cmin=-1, cmax=1, colorbar_title="silhouette"))
    return fig

In [ ]:
plot_hyperparameter_landscape(*landscape_args())

## 04 — Inspect a selected run

The table lists successful main-batch runs in deterministic ID order, not silhouette rank, with at most 20 configurations per zero-based page. Select a numbered row using the two commented choice controls; it binds to this execution and its exact run ID. Alternatively enter an exact run ID directly, which takes precedence. The selected exact ID is reported and retained in `inspection_state`. Failed and incomplete attempts are not choices. A blank or unavailable run selects the family default; an unavailable group selects the largest assigned group. Settings live in the active execution’s editable `inspection_state` dictionary, with automatic resets reported. Uncomment controls to change them, then comment them again before applying different settings or switching executions; uncommented assignments run every time the control cell runs. Select another execution in the active-execution cell before editing its settings. Family profile presets provide defaults for newly initialized states; profile features can also be edited in the active state independently of the two explorer features. Following-week descriptors are fitted information, not independent recovery validation. The default structure view computes display-only PCA from all prepared coordinates; choose 1-based axes or native coordinates in the controls. This projection does not alter fitted assignments. After calculating a matched-medoid configuration in section 05, run the optional controls cell to see its numbered configuration catalog. Use the matched selection example in the inspector controls to choose its method, K and optional restart; use the main selection example to return to the original run catalog. Each source is bound to its retained execution. Inspection otherwise reuses retained data and does not refit clustering or generate seeds.

In [ ]:
# Family profile presets can be edited independently of the explorer features.
inspection_profile_presets = {
    "descriptor": ("price_log_change", "focal_prior_price", "next_prior_price",
        "mean_relative_range", "directional_balance", "peak_timing", "volume_ratio",
        "order_count_ratio", "next_volume_ratio", "peak_day_volume_share"),
    "joint_paths": ("median_log_price_relative", "final_log_price_relative",
        "observed_log_price_change", "observed_log_price_range", "price_peak_timing",
        "mean_activity_ratio", "peak_activity_share", "activity_peak_timing",
        "observed_price_coverage"),
}

def inspection_state_for(execution):
    """One editable state owned by the retained execution; deletion removes it."""
    family = execution.options["family"]
    state = execution.cache.setdefault("inspection-state-v1", dict(
        inspection_run_id=None, inspection_group=None, inspection_competitor=None,
        inspection_features=("price_log_change", "volume_ratio" if family == "descriptor" else "activity_ratio"),
        inspection_activity="volume" if family == "descriptor" else execution.preparation["controls"]["activity_column"],
        inspection_activity_display="relative", inspection_member_view="coverage",
        inspection_challenge_feature="price_log_change", inspection_examples=4,
        inspection_historical_key=None, inspection_profile_view="median",
        inspection_profile_features=inspection_profile_presets[family],
        inspection_projection="prepared_pca", inspection_projection_axes=(1, 2),
        occupancy_level="category", occupancy_category=None, occupancy_item=None,
        occupancy_start=None, occupancy_end=None, optional_mode="input", optional_query_key=None,
        optional_query_row=0, optional_neighbor_rank=1, optional_coordinate=0, optional_neighbors_shown=20))
    # Existing kernel states created before the shortlist retain all their choices.
    state.setdefault("inspection_run_choice", None)
    state.setdefault("inspection_run_page", 0)
    state.setdefault("inspection_matched_source", None)
    state.setdefault("inspection_source_identity", None)
    state.setdefault("inspection_matched_page", 0)
    return state

def inspection_run_catalog(execution):
    """Deterministic successful main-batch choices; number is local to this execution."""
    batch = execution.batch
    entries = []
    for run_id, run in batch.runs.items():
        if run.get("status") != "ok":
            continue
        rep = batch.representations[run["representation_id"]]
        p, rp = rep.get("params", {}), run.get("params", {})
        entries.append(dict(run_id=run_id, method=p.get("method"),
            dimension=p.get("dimension"), neighbors=p.get("neighbors"),
            ae_decay=p.get("weight_decay"), cluster_method=rp.get("method"),
            k=run.get("k", rp.get("n_clusters")), rep_repeat=p.get("repetition"),
            repetition=rp.get("repetition"), seed=run.get("seed")))
    entries.sort(key=lambda entry: str(entry["run_id"]))
    return [dict(choice=index, **entry) for index, entry in enumerate(entries, 1)]


def inspection_matched_catalog(execution):
    """Stable displayed ordinals for retained configurations of this execution."""
    cache = execution.cache.get("matched_medoids", {})
    return [(i, config) for i, config in enumerate(sorted(cache, key=repr), 1)]

def inspection_matched_page(execution, page):
    if isinstance(page, bool) or not isinstance(page, int) or page < 0:
        raise ValueError("Matched page must be a non-negative integer")
    entries = inspection_matched_catalog(execution)
    visible = [dict(choice=i, subset_size=c[0], clusters=str(c[1]), band=c[2],
                    max_iter=c[3], starts=c[4], seed=c[5]) for i, c in entries[page * 20:(page + 1) * 20]]
    return pl.DataFrame(visible, schema={"choice": pl.Int64, "subset_size": pl.Int64,
        "clusters": pl.String, "band": pl.Int64, "max_iter": pl.Int64, "starts": pl.Int64, "seed": pl.Int64})

def inspection_select_matched(execution, choice, method, k, restart=None):
    """Validate before committing; bind the displayed choice to its retained config."""
    if "matched_medoid_inspection" not in globals():
        raise ValueError("Run the section 05 definitions and calculate matched medoids first")
    entries = inspection_matched_catalog(execution)
    if isinstance(choice, bool) or not isinstance(choice, int) or not 1 <= choice <= len(entries):
        raise ValueError("Choose a displayed retained matched configuration")
    config = entries[choice - 1][1]
    matched = matched_medoid_inspection(execution, config, method, k, restart)
    state = inspection_state_for(execution)
    selection = (execution.id, config, method, k, restart)
    if state["inspection_matched_source"] != selection:
        state.update(inspection_group=None, inspection_competitor=None, optional_query_row=0)
    state["inspection_matched_source"] = selection
    state["inspection_source_identity"] = ("matched", matched["run"]["id"])
    return selection

def inspection_select_main(execution):
    state = inspection_state_for(execution)
    if state["inspection_matched_source"] is not None:
        state.update(inspection_group=None, inspection_competitor=None, optional_query_row=0)
    state["inspection_matched_source"] = None
    state["inspection_source_identity"] = None

def inspection_choose_run(execution, choice):
    """Bind a displayed number to both execution identity and its exact run ID."""
    entries = inspection_run_catalog(execution)
    if (isinstance(choice, bool) or not isinstance(choice, int)
            or not 1 <= choice <= len(entries)):
        raise ValueError("Choose a displayed successful run number")
    return (execution.id, choice, entries[choice - 1]["run_id"])


def inspection_run_page(execution, page):
    if isinstance(page, bool) or not isinstance(page, int) or page < 0:
        raise ValueError("Run page must be a non-negative integer")
    entries = inspection_run_catalog(execution)
    visible = [{key: value for key, value in entry.items() if key != "run_id"}
               for entry in entries[page * 20:(page + 1) * 20]]
    return pl.DataFrame(visible, schema={"choice": pl.Int64, "method": pl.String,
        "dimension": pl.Int64, "neighbors": pl.Int64, "ae_decay": pl.Float64,
        "cluster_method": pl.String, "k": pl.Int64, "rep_repeat": pl.Int64,
        "repetition": pl.Int64, "seed": pl.Int64})

In [ ]:
# This is the active execution's editable dictionary, not a copy. Uncomment and edit
# individual settings; rerunning without edits preserves its existing choices.
inspection_state = inspection_state_for(require_active_execution())
# To select a numbered row, uncomment both lines and replace 1 with its choice number.
# inspection_state["inspection_run_choice"] = inspection_choose_run(require_active_execution(), 1)
# inspection_state["inspection_run_id"] = None  # clears any exact-ID override
# Or use an exact successful run ID directly (takes precedence over the numbered choice).
# inspection_state["inspection_run_id"] = "paste-exact-run-id"
# inspection_state["inspection_run_page"] = 0  # zero-based, 20 rows per page
# After section 05, view the matched catalog in the optional input board controls below, then choose a displayed ordinal:
# inspection_select_matched(require_active_execution(), 1, "aligned", 4, None)  # or "dtw"; None = best
# inspection_select_main(require_active_execution())  # return to main-batch choices
# inspection_state["inspection_group"] = None  # largest group
# inspection_state["inspection_competitor"] = None
# inspection_state["inspection_features"] = ("price_log_change", "volume_ratio")
# inspection_state["inspection_activity"] = "volume"  # joint paths use fitted activity
# inspection_state["inspection_activity_display"] = "relative"  # or raw
# inspection_state["inspection_member_view"] = "coverage"  # central, boundary, challenging
# inspection_state["inspection_challenge_feature"] = "price_log_change"
# inspection_state["inspection_examples"] = 4  # coverage: 4; other views: 3–5
# inspection_state["inspection_historical_key"] = None  # exact fitted key
# inspection_state["inspection_profile_view"] = "median"  # or iqr
# inspection_state["inspection_profile_features"] = inspection_profile_presets["descriptor"]
# inspection_state["inspection_projection"] = "prepared_pca"  # or native
# inspection_state["inspection_projection_axes"] = (1, 2)  # 1-based
# inspection_state["occupancy_level"] = "category"  # or item
# inspection_state["occupancy_category"] = None
# inspection_state["occupancy_item"] = None
# inspection_state["occupancy_start"] = None
# inspection_state["occupancy_end"] = None

inspection_run_page(require_active_execution(), inspection_state["inspection_run_page"])

### Selection and aligned inputs

Examples default to four assigned-distance coverage bands. Central means nearest the group median own-center distance; boundary means smallest selected-competitor-minus-own distance, or nearest-other-minus-own when no competitor is selected; challenging means largest absolute deviation from the group median of the selected original feature. Challenging is a heuristic, not narrative validation. Example selection prefers nonoverlapping focal intervals in the same market. An exact historical key routes that fitted observation into the same explorer; it must belong to the selected group. Feature distributions and profiles use whole groups, not just examples. Change feature or activity controls to explore different economic channels.

In [ ]:
import pandas as pd


def _inspection_data(execution):
    """Validate and materialize immutable retained inputs once per execution."""
    cache_key = "inspection-derived-v1"
    if cache_key in execution.cache:
        return execution.cache[cache_key]
    inspection_batch = execution.batch
    inspection_family = execution.options["family"]
    if inspection_family == "descriptor":
        inspection_key_columns = ["type_id", "week_start"]
        inspection_keys = pd.DataFrame(inspection_batch.keys, columns=inspection_key_columns)
        inspection_keys["week_start"] = pd.to_datetime(inspection_keys["week_start"])
        _descriptor_input = execution.preparation
        _fingerprint = inspection_batch.input_metadata.get("source_fingerprint")
        if not _fingerprint or _descriptor_input.get("source_fingerprint") != _fingerprint or source_fingerprint(execution.source["bank"], execution.source["history"]) != _fingerprint:
            raise ValueError("Descriptor batch lacks matching retained source snapshots; refit from the prepared bank")
        inspection_source_history = execution.source["history"]
        _source = execution.source["bank"].to_pandas()
        _source["week_start"] = pd.to_datetime(_source["week_start"])
        inspection_rows = inspection_keys.merge(_source, on=inspection_key_columns, how="left", validate="one_to_one", sort=False)
        inspection_focal_days = 7
        inspection_reference_days = 14
    else:
        _joint = execution.preparation
        paths = _joint["paths"]
        if (_joint.get("population_id") != inspection_batch.population_id
                or tuple(paths.keys.itertuples(index=False, name=None)) != tuple(inspection_batch.keys)
                or tuple(paths.channels) != tuple(inspection_batch.input_metadata.get("channels", ()))
                or not np.array_equal(np.asarray(paths.scales), np.asarray(inspection_batch.input_metadata.get("scales")))
                or not np.array_equal(paths.tensor.reshape(len(paths.keys), -1), inspection_batch.prepared)):
            raise ValueError("Joint preparation does not match the selected population")
        _fingerprint = inspection_batch.input_metadata.get("caller", {}).get("source_fingerprint")
        if not _fingerprint or _joint.get("source_fingerprint") != _fingerprint or source_fingerprint(execution.source["history"]) != _fingerprint:
            raise ValueError("Joint batch lacks matching retained source snapshots; refit from prepared paths")
        inspection_source_history = execution.source["history"]
        inspection_key_columns = list(inspection_batch.input_metadata["key_columns"])
        inspection_keys = pd.DataFrame(inspection_batch.keys, columns=inspection_key_columns)
        inspection_keys["start"] = pd.to_datetime(inspection_keys["start"])
        _selected = _joint["selected"].copy()
        _selected["start"] = pd.to_datetime(_selected["start"])
        inspection_rows = inspection_keys.merge(_selected, on=inspection_key_columns, how="left", validate="one_to_one", sort=False)
        _candidates = _joint["candidates"].copy()
        _candidates["start"] = pd.to_datetime(_candidates["start"])
        inspection_rows = inspection_rows.merge(_candidates[inspection_key_columns + ["price_reference", "activity_reference"]], on=inspection_key_columns, how="left", validate="one_to_one", sort=False)
        from market_analysis.behavior_paths import summarize_joint_paths
        _summary = summarize_joint_paths(paths)
        _summary["start"] = pd.to_datetime(_summary["start"])
        inspection_rows = inspection_rows.merge(_summary, on=inspection_key_columns, how="left", validate="one_to_one", sort=False)
        # Backward-compatible explorer names; endpoint semantics remain exact.
        inspection_rows["price_log_change"] = inspection_rows["observed_log_price_change"]
        inspection_rows["price_range"] = inspection_rows["observed_log_price_range"]
        inspection_rows["activity_ratio"] = inspection_rows["mean_activity_ratio"]
        inspection_focal_days = _joint["controls"]["window"]
        inspection_reference_days = max(14, _joint["controls"]["reference_days"])
    if (len(inspection_rows) != len(inspection_batch.keys)
            or not inspection_rows[inspection_key_columns].equals(inspection_keys[inspection_key_columns])
            or inspection_rows["type_name"].isna().any()):
        raise ValueError("Missing aligned market metadata")
    inspection_history = inspection_source_history.select("region_id", "type_id", "market_date", "vwap", "volume", "order_count", "is_observed").to_pandas()
    inspection_history["market_date"] = pd.to_datetime(inspection_history["market_date"])
    normalized_keys = {tuple(pd.Timestamp(v) if c in ("start", "week_start") else v
                             for c, v in zip(inspection_key_columns, key, strict=True)): i
                       for i, key in enumerate(inspection_batch.keys)}
    result = dict(rows=inspection_rows, history=inspection_history, key_columns=inspection_key_columns,
                  normalized_keys=normalized_keys,
                  focal_days=inspection_focal_days, reference_days=inspection_reference_days)
    execution.cache[cache_key] = result
    return result


def _default_inspection_run(batch, family):
    successful = [r for r in batch.runs.values() if r.get("status") == "ok"]
    if not successful:
        raise ValueError("No successful run in the selected execution")
    def rank(r):
        method = batch.representations[r["representation_id"]]["params"]["method"]
        k = r.get("k", r.get("params", {}).get("n_clusters", 0))
        if family == "descriptor":
            priority = 0 if method == "prepared" and k == 4 else 1 if method == "pca" else 2
        else:
            priority = 0 if method == "aligned" and r.get("params", {}).get("method") == "kmeans" else 1
        return priority, abs(k - 4), k, str(r["id"])
    return min(successful, key=rank)

In [ ]:
def inspection_context():
    """Validate execution-owned controls and attach labels to cached inputs."""
    execution = require_active_execution()
    batch, family = execution.batch, execution.options["family"]
    state = inspection_state_for(execution)
    notices = []
    inspection_run_id = state["inspection_run_id"]
    inspection_group = state["inspection_group"]
    inspection_competitor = state["inspection_competitor"]
    inspection_features = state["inspection_features"]
    inspection_challenge_feature = state["inspection_challenge_feature"]
    inspection_historical_key = state["inspection_historical_key"]
    inspection_activity = state["inspection_activity"]
    inspection_activity_display = state["inspection_activity_display"]
    inspection_member_view = state["inspection_member_view"]
    inspection_examples = state["inspection_examples"]
    source = state["inspection_matched_source"]
    if source is None:
        data = _inspection_data(execution)
        choice = state["inspection_run_choice"]
        selected_id = None
        if choice is not None:
            entries = inspection_run_catalog(execution)
            if (isinstance(choice, tuple) and len(choice) == 3 and choice[0] == execution.id
                    and isinstance(choice[1], int) and not isinstance(choice[1], bool)
                    and 1 <= choice[1] <= len(entries)
                    and entries[choice[1] - 1]["run_id"] == choice[2]):
                selected_id = choice[2]
            else:
                if isinstance(choice, tuple) and len(choice) == 3 and inspection_run_id == choice[2]:
                    inspection_run_id = None
                state["inspection_run_choice"] = None
                notices.append("cleared stale or unavailable run choice")
        if inspection_run_id is None and selected_id is not None:
            inspection_run_id = selected_id
            notices.append(f"selected exact run ID {inspection_run_id}")
        elif inspection_run_id is not None and selected_id != inspection_run_id:
            # An explicit exact ID overrides the numbered choice.
            state["inspection_run_choice"] = None
        run = batch.runs.get(inspection_run_id)
        if run is None or run.get("status") != "ok":
            if inspection_run_id is not None:
                notices.append("cleared unavailable or unsuccessful run ID")
            state["inspection_run_choice"] = None
            run = _default_inspection_run(batch, family)
            inspection_run_id = run["id"]
            notices.append(f"selected run {inspection_run_id[:12]}")
        rep = batch.representations[run["representation_id"]]
        preparation = execution.preparation
        source_identity = ("main", run["id"])
    else:
        if (not isinstance(source, tuple) or len(source) != 5 or source[0] != execution.id):
            raise ValueError("Matched selection belongs to a different execution")
        try:
            adapter = matched_medoid_inspection
        except NameError:
            raise ValueError("Run the section 05 definitions and calculate matched medoids first") from None
        _, config, method, k, restart = source
        matched = adapter(execution, config, method, k, restart)
        batch, run, rep = matched["batch"], matched["run"], matched["rep"]
        run = dict(run, inspection_label=f"matched {method} · K={k} · restart {run['params']['repetition']} · {run['id'][-8:]}")
        preparation = matched["preparation"]
        data = {name: matched[name] for name in ("rows", "history", "key_columns", "focal_days", "reference_days")}
        data["normalized_keys"] = {
            tuple(pd.Timestamp(v) if c in ("start", "week_start") else v
                  for c, v in zip(data["key_columns"], key, strict=True)): i
            for i, key in enumerate(batch.keys)}
        source_identity = ("matched", run["id"])
    query = state["optional_query_key"]
    if query is not None:
        try:
            normalized = tuple(pd.Timestamp(v) if c in ("start", "week_start") else v
                               for c, v in zip(data["key_columns"], query, strict=True))
            present = normalized in data["normalized_keys"]
        except (ValueError, TypeError):
            present = False
        if not present:
            state["optional_query_key"] = None
            notices.append("cleared query key absent from inspection source")
    if tuple(run["keys"]) != tuple(batch.keys):
        raise ValueError("Run keys do not match the selected population")
    labels = np.asarray(run["assignments"])
    if labels.shape != (len(batch.keys),):
        raise ValueError("Run assignments do not match the selected population")
    groups, counts = np.unique(labels, return_counts=True)
    if not len(groups):
        raise ValueError("Selected run has no assigned groups")
    if inspection_group not in groups:
        inspection_group = groups[np.argmax(counts)].item()
        notices.append(f"selected largest group {inspection_group}")
    if inspection_competitor is not None and (inspection_competitor not in groups or inspection_competitor == inspection_group):
        inspection_competitor = None
        notices.append("cleared incompatible competitor")
    available = data["rows"].columns
    if len(inspection_features) != 2 or any(f not in available for f in inspection_features):
        inspection_features = ("price_log_change", "volume_ratio" if family == "descriptor" else "activity_ratio")
        notices.append("reset explorer features")
    if inspection_challenge_feature not in available:
        inspection_challenge_feature = inspection_features[0]
        notices.append("reset challenge feature")
    if inspection_member_view == "typical":
        inspection_member_view = "coverage"
        notices.append("renamed typical to coverage examples")
    if inspection_activity_display not in ("relative", "raw"):
        inspection_activity_display = "relative"
        notices.append("reset activity display to relative")
    if family == "joint_paths":
        fitted_activity = execution.preparation["controls"]["activity_column"]
        if inspection_activity != fitted_activity:
            inspection_activity = fitted_activity
            notices.append(f"selected fitted activity channel {fitted_activity}")
    if inspection_activity not in ("volume", "order_count") or inspection_member_view not in ("central", "coverage", "boundary", "challenging"):
        raise ValueError("Invalid activity channel or member view")
    if inspection_member_view == "coverage" and inspection_examples != 4:
        inspection_examples = 4
        notices.append("coverage uses four bands; reset example count to 4")
    elif inspection_examples not in (3, 4, 5):
        raise ValueError("Choose 3–5 examples for non-coverage views")
    if inspection_historical_key is not None:
        cols = data["key_columns"]
        def normalize(key):
            return tuple(pd.Timestamp(v) if c in ("start", "week_start") else v for c, v in zip(cols, key, strict=True))
        try:
            key = normalize(inspection_historical_key)
            index = data["normalized_keys"][key]
        except (ValueError, TypeError, KeyError):
            inspection_historical_key = None
            notices.append("historical case absent from fitted population; cleared")
        else:
            assigned = labels[index].item()
            if assigned != inspection_group:
                inspection_group = assigned
                notices.append(f"historical case redirected to group {assigned}")
            if inspection_competitor == inspection_group:
                inspection_competitor = None
                notices.append("cleared incompatible competitor")
    rows = data["rows"].copy()
    rows["group"] = labels
    state.update(inspection_run_id=inspection_run_id, inspection_group=inspection_group, inspection_competitor=inspection_competitor, inspection_features=inspection_features, inspection_challenge_feature=inspection_challenge_feature, inspection_historical_key=inspection_historical_key, inspection_activity=inspection_activity, inspection_activity_display=inspection_activity_display, inspection_member_view=inspection_member_view, inspection_examples=inspection_examples)
    state["inspection_source_identity"] = source_identity
    if notices:
        print("Inspection: " + "; ".join(notices) + ".")
    return dict(execution=execution, state=state, batch=batch, family=family, run=run, rep=rep,
                rows=rows, history=data["history"], key_columns=data["key_columns"],
                normalized_keys=data["normalized_keys"], preparation=preparation,
                focal_days=data["focal_days"], reference_days=data["reference_days"])

### Display projection cache

The projection is display-only and does not change clustering. Descriptor embeddings use their first two native coordinates. All joint-path methods use a deterministic two-component PCA projection of the full prepared paths, calculated on demand and cached within the active execution by population and representation; this is not their fitted geometry. Changing groups or plot controls does not refit it. The structure scatter shows at most 4,000 deterministically sampled observations, explicitly labeled; diagnostics and occupancy use their full eligible populations (silhouettes use the batch's retained diagnostic sample).

In [ ]:
def inspection_projection_for(context):
    execution = context["execution"]
    batch, rep = context["batch"], context["rep"]
    axes = tuple(context["state"]["inspection_projection_axes"])
    if len(axes) != 2 or any(not isinstance(a, int) or isinstance(a, bool) or a < 1 for a in axes) or axes[0] == axes[1]:
        raise ValueError("Projection axes must be two distinct positive 1-based integers")
    mode = context["state"]["inspection_projection"]
    if mode not in ("prepared_pca", "native"):
        raise ValueError("Projection must be prepared_pca or native")
    key = ("inspection-projection-v3", execution.id, batch.population_id, rep["id"], mode, axes)
    if key not in execution.cache:
        if mode == "prepared_pca":
            prepared = np.asarray(batch.prepared)
            centered = prepared - prepared.mean(axis=0)
            _, _, vt = np.linalg.svd(centered, full_matrices=False)
            if max(axes) > len(vt):
                raise ValueError(f"Prepared-input PCA has only {len(vt)} available components")
            xy = centered @ vt[np.array(axes) - 1].T
            label = "PCA of complete prepared input (display only; assignments unchanged)"
            axis_labels = tuple(f"Prepared-input PC {a}" for a in axes)
        else:
            embedding = rep.get("embedding")
            if embedding is None or np.asarray(embedding).ndim != 2 or max(axes) > np.asarray(embedding).shape[1]:
                raise ValueError("Selected representation has no requested native coordinates")
            xy = np.asarray(embedding)[:, np.array(axes) - 1].copy()
            label = "Native representation coordinates (display only)"
            axis_labels = tuple(f"Native coordinate {a}" for a in axes)
        execution.cache[key] = (xy, label, axis_labels)
    projection = execution.cache[key]
    if np.asarray(projection[0]).shape != (len(batch.keys), 2):
        raise ValueError("Stale or invalid projection for selected execution")
    return projection

In [ ]:
def inspection_structure_args():
    c = inspection_context()
    return (c["batch"], c["run"], c["rows"], inspection_projection_for(c),
            c["state"]["inspection_profile_features"], c["family"], c["state"]["inspection_profile_view"])


def inspection_explorer_args():
    c = inspection_context()
    return (c["batch"], c["run"], c["rows"], c["history"], c["family"], c["key_columns"],
            c["focal_days"], c["reference_days"], c["state"]["inspection_group"], c["state"]["inspection_competitor"],
            c["state"]["inspection_features"], c["state"]["inspection_activity"], c["state"]["inspection_member_view"], c["state"]["inspection_challenge_feature"],
            c["state"]["inspection_examples"], c["state"]["inspection_historical_key"], c["state"]["inspection_activity_display"],
            c["preparation"]["controls"]["reference_days"] if c["family"] == "joint_paths" else 7)


def occupancy_args():
    c = inspection_context()
    return (c["batch"], c["rows"], c["family"], c["state"]["inspection_group"], c["state"]["occupancy_level"],
            c["state"]["occupancy_category"], c["state"]["occupancy_item"], c["state"]["occupancy_start"], c["state"]["occupancy_end"])


def optional_input_args():
    c = inspection_context()
    batch = c["batch"]
    state = c["state"]
    changes = []
    optional_mode = state["optional_mode"]
    optional_query_key = state["optional_query_key"]
    optional_query_row = state["optional_query_row"]
    optional_coordinate = state["optional_coordinate"]
    optional_neighbor_rank = state["optional_neighbor_rank"]
    optional_neighbors_shown = state["optional_neighbors_shown"]
    if optional_query_key is not None:
        try:
            key = tuple(pd.Timestamp(v) if col in ("start", "week_start") else v
                        for col, v in zip(c["key_columns"], optional_query_key, strict=True))
            exists = key in c["normalized_keys"]
        except (ValueError, TypeError):
            exists = False
        if not exists:
            optional_query_key = None
            changes.append("cleared unavailable query key")
    if not isinstance(optional_query_row, int) or not 0 <= optional_query_row < len(batch.keys):
        optional_query_row = 0
        changes.append("reset query row")
    if not isinstance(optional_coordinate, int) or not 0 <= optional_coordinate < batch.prepared.shape[1]:
        optional_coordinate = 0
        changes.append("reset coordinate")
    if not isinstance(optional_neighbor_rank, int) or not 1 <= optional_neighbor_rank < len(batch.keys):
        optional_neighbor_rank = 1
        changes.append("reset neighbor rank")
    if not isinstance(optional_neighbors_shown, int) or optional_neighbors_shown < 1:
        optional_neighbors_shown = 20
        changes.append("reset neighbor count")
    if optional_mode not in ("input", "geometry", "reconstruction", "ae_history"):
        optional_mode = "input"
        changes.append("reset optional mode")
    if optional_mode in ("reconstruction", "ae_history") and (c["family"] != "descriptor" or c["rep"]["params"]["method"] != "ae"):
        optional_mode = "input"
        changes.append("AE diagnostics unavailable; selected input mode")
    state.update(optional_mode=optional_mode, optional_query_key=optional_query_key, optional_query_row=optional_query_row, optional_coordinate=optional_coordinate, optional_neighbor_rank=optional_neighbor_rank, optional_neighbors_shown=optional_neighbors_shown)
    if changes:
        print("Optional board: " + "; ".join(changes) + ".")
    return (batch, c["rows"], c["preparation"], c["history"], c["rep"], optional_mode,
            optional_query_key, optional_query_row, optional_neighbor_rank, optional_coordinate, optional_neighbors_shown)

### Group structure

The feature profile displays group medians and interquartile ranges after centering on each feature’s population median and scaling by its population IQR; relative values are unavailable when that IQR is zero or undefined. Hover reports the original units and values. The diagnostics panel uses separate labeled axes for retained sampled silhouettes and native center/medoid-distance margins. A small or negative margin indicates a nearby competing center, not independent validation.

In [ ]:
def inspection_group_color(group):
    palette = ("#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf")
    return palette[int(group) % len(palette)]


def plot_inspection_structure(batch, run, rows, projection, features, family, profile_view):
    from plotly.subplots import make_subplots
    if profile_view not in ("median", "iqr"):
        raise ValueError("Profile view must be median or iqr")
    missing = [feature for feature in features if feature not in rows]
    if missing:
        raise ValueError(f"Profile features unavailable: {missing}")
    xy, projection_label, axis_labels = projection
    if len(xy) != len(rows) or len(rows) != len(batch.keys):
        raise ValueError("Projection and keyed rows must match the batch")
    fig = make_subplots(rows=2, cols=2, specs=[[{}, {}], [{}, {"secondary_y": True}]],
        subplot_titles=(projection_label, "Group sizes · all fitted rows", "Group profiles · population-IQR display scale", "Sampled silhouettes / native boundary margin"))
    idx = np.linspace(0, len(rows) - 1, min(4000, len(rows)), dtype=int)
    for group in sorted(rows.group.unique()):
        color = inspection_group_color(group)
        chosen = idx[rows.group.to_numpy()[idx] == group]
        hover = [f"{rows.iloc[i]['type_name']} · {tuple(batch.keys[i])}<br>group {group}" for i in chosen]
        fig.add_trace(go.Scattergl(x=xy[chosen, 0], y=xy[chosen, 1], mode="markers", name=f"group {group}", marker_color=color, text=hover, hovertemplate="%{text}<extra></extra>"), row=1, col=1)
        subset = rows.loc[rows.group == group]
        fig.add_trace(go.Bar(x=[str(group)], y=[len(subset)], marker_color=color, name=f"group {group}", showlegend=False), row=1, col=2)
        sampled = np.asarray(batch.sample_indices)
        group_sample = rows.group.to_numpy()[sampled] == group
        for diagnostic in ("native", "common_input"):
            vals = run[diagnostic].get("sample_silhouette")
            if vals is not None and group_sample.any():
                fig.add_trace(go.Box(x=[f"{group} · {diagnostic}"] * int(group_sample.sum()), y=np.asarray(vals)[group_sample], marker_color=color, line_color=color, name=f"group {group} · {diagnostic}", showlegend=False, boxpoints=False), row=2, col=2, secondary_y=False)
        distances = run["centroid_distances"]
        members = np.flatnonzero(rows.group.to_numpy() == group)
        own = distances[members, rows.group.to_numpy()[members].astype(int)]
        other = distances[members].copy()
        other[np.arange(len(members)), rows.group.to_numpy()[members].astype(int)] = np.inf
        margins = other.min(axis=1) - own
        fig.add_trace(go.Box(x=[f"{group} · native margin"] * len(members), y=margins, marker_color=color, line_color=color, name=f"group {group} · native margin", showlegend=False, boxpoints=False), row=2, col=2, secondary_y=True)
    def feature_role(feature):
        if family == "joint_paths":
            return "observed price coverage" if feature == "observed_price_coverage" else "derived from fitted channels"
        return "used in fitting" if feature in batch.features else "explanatory only"

    # Two heatmaps share one panel; the dropdown switches median versus spread.
    groups = sorted(rows.group.unique())
    medians, spreads, hover = [], [], []
    unavailable = []
    for group in groups:
        median_row, spread_row, hover_row = [], [], []
        for feature in features:
            population = pd.to_numeric(rows[feature], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            values = pd.to_numeric(rows.loc[rows.group == group, feature], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            base = population.median() if len(population) else np.nan
            scale = population.quantile(.75) - population.quantile(.25) if len(population) else np.nan
            valid_scale = np.isfinite(scale) and scale > 0
            q1, median, q3 = (values.quantile(.25), values.median(), values.quantile(.75)) if len(values) else (np.nan, np.nan, np.nan)
            median_row.append((median - base) / scale if valid_scale else np.nan)
            spread_row.append((q3 - q1) / scale if valid_scale else np.nan)
            role = feature_role(feature)
            text = f"group {group}<br>{feature} · {role}<br>median: {median:.6g}<br>Q1: {q1:.6g}<br>Q3: {q3:.6g}<br>available: {len(values)}<br>population median: {base:.6g}<br>population IQR: {scale:.6g}"
            if not valid_scale:
                reason = "zero" if np.isfinite(scale) and scale == 0 else "undefined"
                text += f"<br>Relative value unavailable: population IQR is {reason}."
                unavailable.append((group, feature, text))
            hover_row.append(text)
        medians.append(median_row)
        spreads.append(spread_row)
        hover.append(hover_row)
    # Keep the full feature and fitting role in hover; compact ticks leave room for the panel.
    short_labels = {
        "price_log_change": "Price<br>change", "focal_prior_price": "Focal/prior<br>price",
        "next_prior_price": "Next/prior<br>price", "mean_relative_range": "Mean relative<br>range",
        "directional_balance": "Directional<br>balance", "peak_timing": "Price peak<br>timing",
        "volume_ratio": "Volume<br>ratio", "order_count_ratio": "Order count<br>ratio",
        "next_volume_ratio": "Next volume<br>ratio", "peak_day_volume_share": "Peak-day<br>volume share",
        "median_log_price_relative": "Median price<br>displacement", "final_log_price_relative": "Final price<br>displacement",
        "observed_log_price_change": "Observed price<br>change", "observed_log_price_range": "Observed price<br>range",
        "price_peak_timing": "Price peak<br>timing", "mean_activity_ratio": "Mean activity<br>ratio",
        "peak_activity_share": "Peak activity<br>share", "activity_peak_timing": "Activity peak<br>timing",
        "observed_price_coverage": "Observed price<br>coverage",
    }
    feature_labels = list(features)
    fig.update_xaxes(tickmode="array", tickvals=feature_labels,
                     ticktext=[short_labels.get(f, f.replace("_", " ")) for f in features],
                     tickangle=-45, row=2, col=1)
    # Missing heatmap cells have no hover target; overlay a visible cross.
    if unavailable:
        feature_label = dict(zip(features, feature_labels, strict=True))
        fig.add_trace(go.Scatter(x=[feature_label[f] for _, f, _ in unavailable],
            y=[str(g) for g, _, _ in unavailable], text=[t for _, _, t in unavailable],
            mode="markers", marker=dict(symbol="x", size=10, color="#777777"),
            hovertemplate="%{text}<extra></extra>", showlegend=False, name="Unavailable scale"),
            row=2, col=1)
    heatmap_indices = []
    for name, values in (("median", medians), ("iqr", spreads)):
        heatmap_indices.append(len(fig.data))
        fig.add_trace(go.Heatmap(x=feature_labels, y=[str(g) for g in groups], z=values,
            text=hover, hovertemplate="%{text}<extra></extra>", colorscale="RdBu" if name == "median" else "Viridis",
            zmid=0 if name == "median" else None, colorbar=dict(title="population IQR"),
            visible=(profile_view == name), name=name), row=2, col=1)
    visibility = [trace.visible is not False for trace in fig.data]
    buttons = []
    for index, label in enumerate(("Median", "IQR spread")):
        visible = visibility.copy()
        for j, trace_index in enumerate(heatmap_indices):
            visible[trace_index] = index == j
        buttons.append(dict(label=label, method="update", args=[{"visible": visible}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, x=0, y=.48, xanchor="left", yanchor="bottom")])
    fig.update_xaxes(title_text=axis_labels[0], row=1, col=1)
    fig.update_yaxes(title_text=axis_labels[1], row=1, col=1)
    fig.update_yaxes(title_text="Group ID", row=2, col=1)
    fig.update_yaxes(title_text="Sample silhouette", range=[-1, 1], row=2, col=2, secondary_y=False)
    fig.update_yaxes(title_text="Native distance margin", row=2, col=2, secondary_y=True)
    run_label = run.get("inspection_label", f"Run {run['id'][:12]}")
    fig.update_layout(title=f"{run_label} · deterministic display sample {len(idx):,}/{len(rows):,}",
                      legend=dict(orientation="h", x=0, y=-0.45, xanchor="left", yanchor="top"),
                      margin=dict(b=240), height=1000)
    return fig

In [ ]:
plot_inspection_structure(*inspection_structure_args())

### Selected-group explorer

Coverage examples span four assigned-distance bands with disjoint full fitting support within each market; other views rank central, boundary, or challenging members. An exact historical key replaces the gallery. Histories show observed prices relative to the preceding-week median (descriptors) or retained reference (joint paths), and activity relative to the corresponding reference or in raw units. Examples describe fitted groups, not independent validation.

In [ ]:
def plot_selected_group_explorer(batch, run, rows, history, family, key_columns, focal_days, reference_days, group, competitor, features, activity, member_view, challenge_feature, example_count, historical_key, activity_display, actual_reference_days):
    """Describe a fitted group with support-spaced examples and observed histories."""
    labels = rows.group.to_numpy()
    if group not in labels or (competitor is not None and (competitor == group or competitor not in labels)):
        raise ValueError("Invalid selected group or competitor")
    if member_view not in ("coverage", "central", "boundary", "challenging") or example_count not in (3, 4, 5) or activity_display not in ("relative", "raw"):
        raise ValueError("Invalid example or activity display selection")
    distances = np.asarray(run["centroid_distances"])
    members = np.flatnonzero(labels == group)
    own = distances[members, labels[members].astype(int)]
    date_col = "week_start" if family == "descriptor" else "start"
    def key(i):
        return tuple(str(v) for v in batch.keys[i])
    def market(i):
        row = rows.iloc[i]
        return (row.region_id, row.type_id) if family == "joint_paths" else (row.type_id,)
    def support(i):
        row = rows.iloc[i]
        start = pd.Timestamp(row[date_col])
        if family == "descriptor":
            return start - pd.Timedelta(days=7), start + pd.Timedelta(days=13)
        start, end = pd.Timestamp(row.support_start), pd.Timestamp(row.support_end)
        if pd.isna(start) or pd.isna(end):
            raise ValueError("Joint path is missing its full fitting support")
        return start, end
    markets = {i: market(i) for i in members}
    supports = {i: support(i) for i in members}
    def compatible(i, selected):
        a, b = supports[i]
        for j in selected:
            if markets[i] == markets[j]:
                c, d = supports[j]
                if a <= d and c <= b:
                    return False
        return True
    def normalized(k):
        return tuple(pd.Timestamp(v) if c in ("start", "week_start") else v for c, v in zip(key_columns, k, strict=True))
    shortfall = ""
    if historical_key is not None:
        index = [normalized(k) for k in batch.keys].index(normalized(historical_key))
        if labels[index] != group:
            raise ValueError("Historical case is not in the selected group")
        chosen = [index]
        criterion = "exact historical case"
    elif member_view == "coverage":
        # Rank bands use key ordering for equal distances. Search jointly so an early
        # choice cannot sacrifice a later band or market diversity.
        ordered = sorted(members, key=lambda i: (distances[i, int(labels[i])], key(i)))
        bands = [list(a) for a in np.array_split(ordered, 4)]
        candidates = []
        for band in bands:
            middle = (len(band) - 1) / 2
            candidates.append(sorted(((i, abs(rank - middle)) for rank, i in enumerate(band)), key=lambda p: (p[1], key(p[0]))))
        suffix_markets = [set() for _ in range(5)]
        for b in range(3, -1, -1):
            suffix_markets[b] = suffix_markets[b + 1] | {markets[i] for i, _ in candidates[b]}
        suffix_penalty = [sum(band[0][1] for band in candidates[b:] if band) for b in range(5)]
        best_score, best_selection = None, []
        def search_band(band_no, selected, penalty):
            nonlocal best_score, best_selection
            remaining = 4 - band_no
            if best_score is not None and len(selected) + remaining < -best_score[0]:
                return
            if best_score is not None and len(selected) + remaining == -best_score[0]:
                seen = {markets[i] for i in selected}
                possible = suffix_markets[band_no]
                unseen_count = len(possible) - sum(m in possible for m in seen)
                max_diversity = len(seen) + min(remaining, unseen_count)
                if max_diversity < -best_score[1]:
                    return
                if max_diversity == -best_score[1]:
                    lower_penalty = penalty + suffix_penalty[band_no]
                    if lower_penalty > best_score[2]:
                        return
            if band_no == 4:
                score = (-len(selected), -len({markets[i] for i in selected}), penalty, tuple(key(i) for i in selected))
                if best_score is None or score < best_score:
                    best_score, best_selection = score, selected.copy()
                return
            seen = {markets[i] for i in selected}
            for already_seen in (False, True):
                for i, distance in candidates[band_no]:
                    if (markets[i] in seen) == already_seen and compatible(i, selected):
                        search_band(band_no + 1, selected + [i], penalty + distance)
            search_band(band_no + 1, selected, penalty)
        search_band(0, [], 0)
        chosen = best_selection
        criterion = "coverage examples · four assigned-distance rank bands"
        if len(chosen) < 4:
            shortfall = f" · {4-len(chosen)} band(s) empty or blocked by full-support spacing"
    else:
        if member_view == "central":
            rank = own
            criterion = "central · smallest assigned distance"
        elif member_view == "boundary":
            if competitor is not None:
                rank = distances[members, int(competitor)] - own
            else:
                other = distances[members].copy()
                other[np.arange(len(members)), labels[members].astype(int)] = np.inf
                rank = other.min(axis=1) - own
            criterion = "boundary · smallest competing-minus-assigned distance"
        else:
            values = pd.to_numeric(rows.iloc[members][challenge_feature], errors="coerce").to_numpy()
            if not np.isfinite(values).any():
                raise ValueError("Challenging feature has no finite values in the selected group")
            rank = -np.abs(values - np.nanmedian(values))
            rank = np.where(np.isfinite(rank), rank, np.inf)
            criterion = f"challenging · largest deviation from group median of {challenge_feature}"
        ordered = sorted(zip(members, rank), key=lambda pair: (pair[1], key(pair[0])))
        chosen = []
        while len(chosen) < example_count:
            admissible = [(i, score) for i, score in ordered if i not in chosen and compatible(i, chosen) and np.isfinite(score)]
            if not admissible:
                break
            seen = {markets[i] for i in chosen}
            chosen.append(min(admissible, key=lambda pair: (markets[pair[0]] in seen, pair[1], key(pair[0])))[0])
        if len(chosen) < example_count:
            shortfall = f" · {example_count-len(chosen)} unavailable under full-support spacing"
    fig = make_subplots(rows=2, cols=2, subplot_titles=(f"{features[0]} · whole groups", f"{features[1]} · whole groups", "Observed price · reference median = 100", f"Observed {activity} · {'reference mean = 1' if activity_display == 'relative' else 'raw'}"))
    for col, feature in enumerate(features, 1):
        for label, mask in ((f"group {group}", labels == group), (f"group {competitor}" if competitor is not None else "all other groups", labels == competitor if competitor is not None else labels != group)):
            values = pd.to_numeric(rows.loc[mask, feature], errors="coerce").dropna()
            color = inspection_group_color(group) if label == f"group {group}" else (inspection_group_color(competitor) if competitor is not None else "#777777")
            fig.add_trace(go.Box(y=values, name=label, marker_color=color, line_color=color, showlegend=(col == 1), boxpoints=False), row=1, col=col)
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
    for j, i in enumerate(chosen):
        r = rows.iloc[i]
        start = pd.Timestamp(r[date_col])
        end = start + pd.Timedelta(days=focal_days - 1)
        ref_start, ref_end = start - pd.Timedelta(days=actual_reference_days), start - pd.Timedelta(days=1)
        mask = (history.type_id == r.type_id) & history.market_date.between(start - pd.Timedelta(days=reference_days), end + pd.Timedelta(days=14))
        if family == "joint_paths":
            mask &= history.region_id == r.region_id
        h = history.loc[mask].sort_values("market_date")
        prior = h.loc[h.market_date.between(ref_start, ref_end)]
        observed = prior.is_observed.fillna(False).to_numpy(dtype=bool)
        prior_prices = pd.to_numeric(prior.vwap, errors="coerce").to_numpy(dtype=float)
        valid_price = observed & np.isfinite(prior_prices) & (prior_prices > 0)
        prior_activity = pd.to_numeric(prior[activity], errors="coerce").to_numpy(dtype=float)
        valid_activity = np.isfinite(prior_activity) & (prior_activity >= 0)
        if family == "descriptor":
            price_ref = np.median(prior_prices[valid_price]) if valid_price.any() else np.nan
            activity_ref = prior_activity.mean() if len(prior_activity) and valid_activity.all() else np.nan
            definition = "preceding-week observed median price and mean daily activity"
        else:
            price_ref, activity_ref = r.price_reference, r.activity_reference
            definition = "retained preparation price and activity references"
        last_prior = prior_prices[np.flatnonzero(valid_price)[-1]] if valid_price.any() else np.nan
        prices = pd.to_numeric(h.vwap, errors="coerce").to_numpy(dtype=float)
        prices = np.where(h.is_observed.fillna(False).to_numpy(dtype=bool) & np.isfinite(prices) & (prices > 0), prices, np.nan)
        raw_activity = pd.to_numeric(h[activity], errors="coerce").to_numpy(dtype=float)
        indexed = prices / price_ref * 100 if np.isfinite(price_ref) and price_ref > 0 else np.full(len(prices), np.nan)
        relative_activity = raw_activity / activity_ref if np.isfinite(activity_ref) and activity_ref > 0 else np.full(len(raw_activity), np.nan)
        displayed_activity = relative_activity if activity_display == "relative" else raw_activity
        rel = (h.market_date - start).dt.days
        identifier = f"{r.type_name} · {start.date()}"
        detail = (f"type ID: {r.type_id}<br>{definition}<br>reference: {ref_start.date()} to {ref_end.date()}"
                  f"<br>applicable days: {len(prior)}; valid observed price days: {valid_price.sum()}"
                  f"<br>price reference: {price_ref:.6g}; activity reference: {activity_ref:.6g}"
                  f"<br>last valid prior price (movement anchor): {last_prior:.6g}")
        fig.add_trace(go.Scatter(x=rel, y=indexed, mode="lines", name=identifier, legendgroup=identifier, line_color=colors[j], connectgaps=False, customdata=prices, meta=detail, hovertemplate="day %{x}<br>price index %{y:.3f}<br>raw price %{customdata}<br>%{meta}<extra>%{fullData.name}</extra>"), row=2, col=1)
        fig.add_trace(go.Scatter(x=rel, y=displayed_activity, mode="lines", name=identifier, legendgroup=identifier, line_color=colors[j], showlegend=False, connectgaps=False, customdata=raw_activity, meta=detail, hovertemplate="day %{x}<br>displayed activity %{y}<br>raw activity %{customdata}<br>%{meta}<extra>%{fullData.name}</extra>"), row=2, col=2)
    for col in (1, 2):
        fig.add_vrect(x0=0, x1=focal_days-1, fillcolor="gray", opacity=.12, line_width=0, row=2, col=col)
        fig.update_xaxes(title_text="Days relative to focal start; shaded = focal interval", row=2, col=col)
    fig.add_hline(y=100, line_dash="dot", line_color="gray", row=2, col=1)
    if activity_display == "relative":
        fig.add_hline(y=1, line_dash="dot", line_color="gray", row=2, col=2)
    fig.update_layout(title=f"Group {group} · {criterion} · {len(chosen)} example(s){shortfall}",
                      legend=dict(orientation="h", x=0, y=-0.20, xanchor="left", yanchor="top"),
                      margin=dict(b=150), height=950)
    return fig

In [ ]:
plot_selected_group_explorer(*inspection_explorer_args())

### Markets and calendar occupancy

Counts and denominators use exactly the selected fitted population, including a matched subset if that is the chosen batch; rejected candidates and unrelated populations are excluded. The selected group is the numerator. Category, item and inclusive date filters apply to both numerator and denominator. Toggle item versus category to reconsider market concentration; calendar occupancy is grouped by month. Occupancy counts overlapping windows and is not an independent event frequency.

In [ ]:
# Occupancy controls are shared with the per-execution inspection settings above.

In [ ]:
def plot_group_occupancy(batch, rows, family, group, level, category, item, start, end):
    if level not in ("category", "item") or group not in rows.group.to_numpy():
        raise ValueError("Invalid occupancy level or group")
    date_col = "week_start" if family == "descriptor" else "start"
    frame = rows.copy()
    if category is not None:
        frame = frame.loc[frame.study_category == category]
    if item is not None:
        frame = frame.loc[frame.type_id == item]
    if start is not None:
        frame = frame.loc[frame[date_col] >= pd.Timestamp(start)]
    if end is not None:
        frame = frame.loc[frame[date_col] <= pd.Timestamp(end)]
    frame = frame.copy()
    frame["selected"] = frame.group == group
    frame["market_label"] = frame.study_category if level == "category" else frame.type_name.astype(str) + " · " + frame.type_id.astype(str)
    frame["month"] = pd.to_datetime(frame[date_col]).dt.to_period("M").dt.to_timestamp()
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"{level.title()} occupancy", "Calendar occupancy"))
    for col, key in ((1, "market_label"), (2, "month")):
        agg = frame.groupby(key, sort=True).agg(numerator=("selected", "sum"), denominator=("selected", "size")).reset_index()
        share = agg.numerator / agg.denominator
        fig.add_trace(go.Bar(x=agg[key], y=share, customdata=np.column_stack((agg.numerator, agg.denominator)), hovertemplate="%{x}<br>group: %{customdata[0]}<br>eligible population: %{customdata[1]}<br>share: %{y:.1%}<extra></extra>", showlegend=False), row=1, col=col)
        fig.update_yaxes(title_text="Selected group / eligible fitted windows", tickformat=".0%", range=[0, 1], row=1, col=col)
    if frame.empty:
        fig.add_annotation(text="No eligible windows match these filters", x=.5, y=.5, xref="paper", yref="paper", showarrow=False)
    fig.update_layout(title=f"Population {batch.population_id[:12]} · group {group} · {len(frame):,} eligible windows", height=500)
    return fig

In [ ]:
plot_group_occupancy(*occupancy_args())

## 05 — Shortlist checks and findings

The optional seed check compares already-fitted repetitions with matching configurations and keyed populations. ARI is invariant to group-label permutations; neither it nor raw label agreement establishes economic validity. One repetition cannot measure seed stability.

The separate matched-medoid mode rebuilds a deterministic subset of at most 2,000 eligible paths from the retained joint execution's source history and preparation controls. Pooled channel scales are fitted on that subset, as in joint-path preparation, and both distances use exactly the same prepared paths and paired initial medoid keys. Aligned and dependent-DTW objectives have different meanings and are not a cross-distance ranking. This isolates a distance change between medoid pipelines, unlike the existing aligned K-means versus DTW-medoids complete-pipeline comparison. Calendar transfer applies a training-period pipeline to heldout observations; calendar rediscovery fits periods independently. Both use complete calendar support and describe groups without classification accuracy or forced label matching. Window-origin applies the retained fixed pipeline to shifted windows and compares overlapping same-market episodes and their phase; descriptor following-week context is fitted information, not independent validation. Targeted-input checks fit caller-declared treatments on common eligible keys and identical support, with paired seeds; labels and distances remain run-specific, not accuracy or a cross-geometry quality ranking.

In [ ]:
shortlist_mode = "calendar_transfer"  # calendar_transfer, calendar_rediscovery, window_origin, seed, matched_medoids, targeted_inputs
shortlist_show = False  # display the selected optional board
shortlist_notes = ""

### Seed check controls

Run this control cell only when using this mode.

In [ ]:
shortlist_run_ids = ()  # two or more matching successful IDs for the seed board

### Matched medoids controls

Run this control cell only when using this mode.

In [ ]:
matched_calculate = False
matched_subset_size = 2000
matched_clusters = (4, 8)
matched_dtw_band = 2
matched_max_iter = 100
matched_starts = 3
matched_seed = 42  # controls both deterministic subset and paired starts
matched_display_k = 4
matched_display_restart = None  # None: each method's best converged; otherwise restart index

### Calendar transfer and rediscovery controls

Run this control cell only when using this mode.

In [ ]:
calendar_calculate = False  # explicit calculation only
calendar_run_id = None  # full ID of one successful run in the active execution
calendar_training_period = None  # (inclusive start, inclusive end); required
calendar_heldout_period = None  # (inclusive start, inclusive end); required
calendar_profile_feature = None  # None selects first fitted descriptor or observed joint range

### Window origin controls

Run this control cell only when using this mode.

In [ ]:
origin_run_id = None  # successful retained run ID
origin_shift_days = None  # explicit integer; no study default
origin_min_overlap_days = None  # explicit positive integer
origin_calculate = False
origin_profile_feature = None  # None: first available fitted descriptor or joint summary
origin_pair_index = 0  # zero-based row in deterministic matched-pair order

### Targeted inputs controls

Run this control cell only when using this mode.

In [ ]:
target_run_id = None  # successful retained run ID
target_experiment_label = None  # explicit cache label
target_treatments = {}  # explicit helper-format declarations; no implicit treatments
target_split_column = None
target_split_values = None  # both None: full common population
target_baseline_split = None  # keyed pandas frame when split is supplied
target_seed_namespace = None  # explicit nonempty deterministic namespace
target_calculate = False
target_profile_feature = None  # choose an original-unit column common to displayed treatments

### Shortlist calculation and inspection definitions

Run these definition cells once before calculating or rendering a shortlist check.

In [ ]:
def matched_medoid_config(subset_size, clusters, band, max_iter, starts, seed):
    clusters = tuple(clusters)
    ints = (subset_size, band, max_iter, starts, seed, *clusters)
    if (any(not isinstance(v, int) or isinstance(v, bool) for v in ints)
            or not 1 <= subset_size <= 2000 or not clusters or len(set(clusters)) != len(clusters)
            or min(clusters) < 1 or band < 0 or max_iter < 1 or starts < 1 or seed < 0):
        raise ValueError("Invalid matched-medoid controls (subset must be 1–2000)")
    return (subset_size, clusters, band, max_iter, starts, seed)


def matched_medoid_plan(execution, config):
    if execution.options.get("family") != "joint_paths":
        raise ValueError("Matched medoids require a retained joint-path execution")
    subset_size, clusters, band, max_iter, starts, seed = config
    eligible = int(execution.preparation["candidates"].eligible.sum())
    selected = min(eligible, subset_size)
    if not selected or max(clusters) > selected:
        raise ValueError("Selected eligible population must contain at least max(K) paths")
    return {"eligible": eligible, "selected": selected, "distance_matrices": 2,
            "restart_attempts": 2 * len(clusters) * starts,
            "pairwise_bytes_each": selected * selected * 8}


def calculate_matched_medoids(execution, config):
    """Explicit calculation; retain prepared paths and all results per configuration."""
    from market_analysis.behavior_workbench import compare_path_medoids
    plan = matched_medoid_plan(execution, config)
    cache = execution.cache.setdefault("matched_medoids", {})
    if config in cache:
        print("Matched-medoid configuration already cached; no computation performed.")
        return cache[config]
    subset_size, clusters, band, max_iter, starts, seed = config
    controls = execution.preparation["controls"]
    columns = [*controls["market_columns"], controls["date_column"], controls["price_column"],
               controls["activity_column"], controls["observed_column"]]
    history = execution.source["history"].select(*columns).to_pandas()
    paths = prepare_joint_paths(history, market_columns=tuple(controls["market_columns"]),
        date_column=controls["date_column"], price_column=controls["price_column"],
        activity_column=controls["activity_column"], observed_column=controls["observed_column"],
        window=controls["window"], stride=controls["stride"], reference_days=controls["reference_days"],
        min_price_reference=controls["min_price_reference"], subset_size=subset_size, seed=seed,
        origin_shift_days=execution.preparation["paths"].origin_shift_days)
    if len(paths.keys) != plan["selected"]:
        raise ValueError("Rebuilt eligible population differs from retained preparation")
    result = compare_path_medoids(paths, clusters=clusters, dtw_band=band,
                                  max_iter=max_iter, starts=starts, seed=seed)
    cache[config] = {"paths": paths, "result": result, "config": config,
                     "execution_id": execution.id, "preparation_controls": dict(controls)}
    for method, distance in result["distances"].items():
        seconds = sum(r["fit_seconds"] for k in clusters for r in result["groups"][k][method]["restarts"])
        print(f"{method}: distance {distance['status']} in {distance['fit_seconds']:.3f} s; "
              f"restart compute {seconds:.3f} s; failures and convergence are shown in the board.")
    return cache[config]

In [ ]:
def calendar_shortlist_config(execution, mode, run_id, training_period, heldout_period):
    from market_analysis.behavior_robustness import calendar_masks
    if mode not in ("calendar_transfer", "calendar_rediscovery"):
        raise ValueError("Choose calendar_transfer or calendar_rediscovery")
    if training_period is None or heldout_period is None:
        raise ValueError("Enter explicit inclusive training and heldout periods")
    calendar_masks(pd.DataFrame({"support_start": [], "support_end": []}), training_period=training_period, heldout_period=heldout_period)
    batch = execution.batch
    if run_id not in batch.runs or batch.runs[run_id]["status"] != "ok":
        raise ValueError("Select one successful run ID from the active execution")
    run = batch.runs[run_id]
    rep = batch.representations[run["representation_id"]]
    family, method = execution.options.get("family"), rep["params"]["method"]
    if (family == "descriptor" and method not in ("prepared", "pca", "umap", "ae") or
        family == "joint_paths" and method not in ("aligned", "dtw") or family not in ("descriptor", "joint_paths")):
        raise ValueError("Unsupported run family or representation for calendar checks")
    periods = tuple(tuple(pd.Timestamp(x).isoformat() for x in p) for p in (training_period, heldout_period))
    return (mode, run_id, periods, repr(execution.settings), repr(rep["params"]), repr(run["params"]))


def calculate_calendar_shortlist(execution, config):
    from market_analysis.behavior_robustness import calendar_descriptors, calendar_joint_paths
    cache = execution.cache.setdefault("calendar_checks", {})
    if config in cache:
        print("Calendar configuration already cached; no computation performed.")
        return cache[config]
    mode, run_id, periods, *_ = config
    batch, settings = execution.batch, execution.settings
    run = batch.runs[run_id]
    rep = batch.representations[run["representation_id"]]
    p, method = rep["params"], rep["params"]["method"]
    options = dict(seed=batch.master_seed, repetitions=1, clusters=(run["k"],))
    if execution.options["family"] == "descriptor":
        options.update(methods=(method,), dimensions=(p["dimension"],), pca_dimensions=None,
                       umap_dimensions=None, ae_dimensions=None, ae_enabled=(method == "ae"))
        if method == "umap":
            options.update(neighbors=(p["neighbors"],), umap_min_dist=p["min_dist"])
        if method == "ae":
            options.update(ae_weight_decays=(p["weight_decay"],))
        narrow = replace(settings, **options)
        prep = execution.preparation
        keys = list(execution.options["input_metadata"]["key_columns"])
        frame = prep["bank"].to_pandas().copy()
        frame["support_start"] = pd.to_datetime(frame["week_start"]) - pd.Timedelta(days=7)
        frame["support_end"] = pd.to_datetime(frame["week_start"]) + pd.Timedelta(days=13)
        result = calendar_descriptors(frame, key_columns=keys, features=list(batch.features), week_column="week_start",
            profile_columns=list(batch.features), training_period=periods[0], heldout_period=periods[1],
            settings=narrow, mode=mode.removeprefix("calendar_"))
    else:
        narrow = replace(settings, **options)
        include_dtw = method == "dtw"
        band = p["band"] if include_dtw else execution.options["dtw_band"]
        iterations = run["params"]["max_iter"] if include_dtw else execution.options["medoid_max_iter"]
        result = calendar_joint_paths(execution.preparation["paths"], training_period=periods[0], heldout_period=periods[1],
            settings=narrow, mode=mode.removeprefix("calendar_"), include_dtw=include_dtw,
            dtw_band=band, medoid_max_iter=iterations)
    chosen = {}
    for period, fitted in result["batches"].items():
        candidates = [r for r in fitted.runs.values() if r["k"] == run["k"] and
                      fitted.representations[r["representation_id"]]["params"]["method"] == method]
        if len(candidates) != 1 or candidates[0]["status"] != "ok":
            reason = candidates[0].get("reason") if candidates else "missing requested run"
            raise ValueError(f"{period} calendar fit failed or skipped: {reason}")
        chosen[period] = candidates[0]["id"]
    if mode == "calendar_transfer":
        inference = result["transfer"][chosen["training"]]
        if inference["status"] != "ok":
            raise ValueError(f"Heldout inference failed: {inference.get('reason')}")
    retained = {"result": result, "chosen": chosen, "config": config, "family": execution.options["family"],
                "features": tuple(batch.features), "execution_id": execution.id}
    cache[config] = retained
    return retained

In [ ]:
def calculate_window_origin(execution, run_id, shift_days, min_overlap_days):
    """Apply one retained run to shifted windows without refitting; retain keyed memberships."""
    from copy import copy
    from market_analysis.behavior_paths import summarize_joint_paths
    from market_analysis.behavior_robustness import _descriptor_transfer, _path_transfer, match_episode_windows

    if not isinstance(shift_days, int) or isinstance(shift_days, bool):
        raise ValueError("shift_days must be an explicit integer")
    if not isinstance(min_overlap_days, int) or isinstance(min_overlap_days, bool) or min_overlap_days < 1:
        raise ValueError("min_overlap_days must be an explicit positive integer")
    batch = execution.batch
    family = execution.options.get("family")
    if family not in ("descriptor", "joint_paths"):
        raise ValueError("Unsupported window-origin family")
    if run_id not in batch.runs or batch.runs[run_id].get("status") != "ok":
        raise ValueError("Select a successful retained run")
    run = batch.runs[run_id]
    rep_id = run["representation_id"]
    rep = batch.representations[rep_id]
    method = rep["params"]["method"]
    if method not in (("prepared", "pca", "umap", "ae") if family == "descriptor" else ("aligned", "dtw")):
        raise ValueError("Selected representation does not match the input family")
    if tuple(run["keys"]) != tuple(batch.keys):
        raise ValueError("Selected run keys do not match the retained population")
    config = (run_id, shift_days, min_overlap_days)
    cache = execution.cache.setdefault("window_origin", {})
    if config in cache:
        return cache[config]

    # Restrict inference to the selected run; retain model and transform ownership.
    view = copy(batch)
    view.runs = {run_id: run}
    view.representations = {rep_id: rep}
    source_history = execution.source["history"].clone()
    original_paths = shifted_paths = None
    if family == "descriptor":
        key_columns = ["type_id", "week_start"]
        market_columns = ["type_id"]
        original_keys = pd.DataFrame(list(batch.keys), columns=key_columns)
        bank = execution.preparation["bank"].to_pandas()
        original_keys["week_start"] = pd.to_datetime(original_keys["week_start"])
        bank["week_start"] = pd.to_datetime(bank["week_start"])
        original_profiles = original_keys.merge(bank, on=key_columns, how="left", validate="one_to_one", sort=False)
        if original_profiles["week_end"].isna().any():
            raise ValueError("Retained descriptor profiles are missing fitted keys")
        shifted_bank = build_feature_bank(source_history, origin_shift_days=shift_days)
        if shifted_bank.is_empty():
            raise ValueError("No eligible shifted descriptor windows")
        shifted_profiles = shifted_bank.to_pandas()
        raw = shifted_profiles.loc[:, list(batch.features)].to_numpy(dtype=float)
        eligible = np.isfinite(raw).all(axis=1)
        shifted_profiles = shifted_profiles.loc[eligible].reset_index(drop=True)
        raw = raw[eligible]
        if not len(raw):
            raise ValueError("No finite shifted descriptor windows")
        inferred = _descriptor_transfer(view, raw)[run_id]
        for frame in (original_profiles, shifted_profiles):
            frame["episode_start"] = pd.to_datetime(frame["week_start"])
            frame["episode_end"] = frame["episode_start"] + pd.Timedelta(days=6)
            frame["support_start"] = frame["episode_start"] - pd.Timedelta(days=7)
            frame["support_end"] = frame["episode_start"] + pd.Timedelta(days=13)
    else:
        original_paths = execution.preparation["paths"]
        controls = execution.preparation["controls"]
        key_columns = list(original_paths.keys.columns)
        market_columns = list(controls["market_columns"])
        columns = [*market_columns, controls["date_column"], controls["price_column"],
                   controls["activity_column"], controls["observed_column"]]
        history = source_history.select(*columns).to_pandas()
        shifted_paths = prepare_joint_paths(history, market_columns=tuple(market_columns),
            date_column=controls["date_column"], price_column=controls["price_column"],
            activity_column=controls["activity_column"], observed_column=controls["observed_column"],
            window=controls["window"], stride=controls["stride"], reference_days=controls["reference_days"],
            min_price_reference=controls["min_price_reference"], scales=original_paths.scales,
            origin_shift_days=original_paths.origin_shift_days + shift_days)
        if not len(shifted_paths.keys):
            raise ValueError("No eligible shifted joint-path windows")
        if tuple(original_paths.keys.itertuples(index=False, name=None)) != tuple(batch.keys):
            raise ValueError("Retained joint-path keys do not match the fitted population")
        def profiles(paths):
            return paths.keys.merge(summarize_joint_paths(paths), on=key_columns,
                                    how="left", validate="one_to_one", sort=False)
        original_profiles, shifted_profiles = profiles(original_paths), profiles(shifted_paths)
        inferred = _path_transfer(view, shifted_paths.transformed)[run_id]
        for frame in (original_profiles, shifted_profiles):
            frame["episode_start"] = pd.to_datetime(frame["start"])
            frame["episode_end"] = frame["episode_start"] + pd.Timedelta(days=controls["window"] - 1)
    if inferred["status"] != "ok":
        raise ValueError(f"Fixed-pipeline window-origin inference failed: {inferred.get('reason')}")
    original_assignments = original_profiles[key_columns].copy()
    original_assignments["group"] = np.asarray(run["assignments"])
    shifted_assignments = shifted_profiles[key_columns].copy()
    shifted_assignments["group"] = inferred["assignments"]
    matches = match_episode_windows(original_profiles, shifted_profiles, key_columns=key_columns,
                                    market_columns=market_columns, min_overlap_days=min_overlap_days)
    retained = {"execution_id": execution.id, "family": family, "run_id": run_id,
                "shift_days": shift_days, "min_overlap_days": min_overlap_days,
                "key_columns": tuple(key_columns), "market_columns": tuple(market_columns),
                "original_profiles": original_profiles, "shifted_profiles": shifted_profiles,
                "original_assignments": original_assignments, "shifted_assignments": shifted_assignments,
                "matches": matches, "inference": inferred, "source_history": source_history,
                "original_paths": original_paths, "shifted_paths": shifted_paths}
    cache[config] = retained
    return retained

In [ ]:
def window_origin_config(execution, run_id, shift_days, min_overlap_days):
    """Validate explicit controls without preparing data or running inference."""
    if execution.options.get("family") not in ("descriptor", "joint_paths"):
        raise ValueError("Window origin requires a descriptor or joint-path execution")
    if run_id not in execution.batch.runs or execution.batch.runs[run_id].get("status") != "ok":
        raise ValueError("Select a successful run ID from the active execution")
    if not isinstance(shift_days, int) or isinstance(shift_days, bool):
        raise ValueError("Enter an explicit integer origin shift in days")
    if not isinstance(min_overlap_days, int) or isinstance(min_overlap_days, bool) or min_overlap_days < 1:
        raise ValueError("Enter an explicit positive minimum overlap in days")
    run = execution.batch.runs[run_id]
    rep = execution.batch.representations[run["representation_id"]]
    methods = ("prepared", "pca", "umap", "ae") if execution.options["family"] == "descriptor" else ("aligned", "dtw")
    if rep["params"]["method"] not in methods or tuple(run["keys"]) != tuple(execution.batch.keys):
        raise ValueError("The selected run is incompatible with this execution")
    return (run_id, shift_days, min_overlap_days)

In [ ]:
def targeted_equal(a, b):
    """Compare a declaration to its retained snapshot without hashing or fitting."""
    from dataclasses import fields, is_dataclass
    if type(a) is not type(b):
        return False
    if isinstance(a, pd.DataFrame):
        return a.equals(b)
    if isinstance(a, np.ndarray):
        return np.array_equal(a, b, equal_nan=True)
    if is_dataclass(a) and not isinstance(a, type):
        return all(targeted_equal(getattr(a, f.name), getattr(b, f.name)) for f in fields(a))
    if isinstance(a, dict):
        return a.keys() == b.keys() and all(targeted_equal(a[k], b[k]) for k in a)
    if isinstance(a, (tuple, list)):
        return len(a) == len(b) and all(targeted_equal(x, y) for x, y in zip(a, b))
    return bool(a == b)


def targeted_controls(execution, run_id, label, treatments, split_column, split_values, baseline_split, seed_namespace):
    if not isinstance(label, str) or not label.strip() or not isinstance(seed_namespace, str) or not seed_namespace.strip():
        raise ValueError("Enter an experiment label and deterministic seed namespace")
    if not isinstance(treatments, dict) or not treatments or "baseline" in treatments:
        raise ValueError("Declare at least one named treatment distinct from baseline")
    if (split_column is None) != (split_values is None):
        raise ValueError("Split column and values must both be supplied or both be None")
    if split_column is None and baseline_split is not None:
        raise ValueError("Baseline split must be None for full common population")
    if split_column is not None and (baseline_split is None or len(split_values) != 2 or split_values[0] == split_values[1]):
        raise ValueError("Supply a keyed baseline split and two distinct split values")
    batch = execution.batch
    if run_id not in batch.runs or batch.runs[run_id].get("status") != "ok":
        raise ValueError("Select a successful run in the active execution")
    run = batch.runs[run_id]
    if tuple(run["keys"]) != tuple(batch.keys):
        raise ValueError("Run keys do not match the retained population")
    method = batch.representations[run["representation_id"]]["params"]["method"]
    family = execution.options.get("family")
    if method not in (("prepared", "pca", "umap", "ae") if family == "descriptor" else ("aligned", "dtw") if family == "joint_paths" else ()):
        raise ValueError("Run representation does not match the execution family")
    if family == "descriptor" and method == "ae":
        raise ValueError("targeted AE is unsupported: its internal split needs shared preprocessing")
    return {"run_id":run_id, "label":label, "treatments":treatments, "split_column":split_column,
            "split_values":split_values, "baseline_split":baseline_split, "seed_namespace":seed_namespace}

In [ ]:
def calculate_targeted_inputs(execution, declaration):
    """Fit the explicitly declared common-population treatments once per fingerprint."""
    from copy import deepcopy
    import hashlib
    import pickle
    from market_analysis.behavior_robustness import targeted_descriptors, targeted_joint_paths
    batch, run_id = execution.batch, declaration["run_id"]
    run = batch.runs[run_id]
    rep = batch.representations[run["representation_id"]]
    p, method = rep["params"], rep["params"]["method"]
    options = dict(seed=batch.master_seed, repetitions=1, clusters=(run["k"],), kmeans_n_init=run["params"].get("n_init", execution.settings.kmeans_n_init))
    if execution.options["family"] == "descriptor":
        options.update(methods=(method,), dimensions=(p["dimension"],), pca_dimensions=None,
                       umap_dimensions=None, ae_dimensions=None, ae_enabled=False)
        if method == "umap":
            options.update(neighbors=(p["neighbors"],), umap_min_dist=p["min_dist"])
    settings = replace(execution.settings, **options)
    # Include complete declarations, retained baseline, actual selected fit configuration and split.
    baseline = execution.preparation["bank"] if execution.options["family"] == "descriptor" else execution.preparation["paths"]
    snapshot = deepcopy(declaration)
    fingerprint = hashlib.sha256(pickle.dumps((snapshot, baseline, settings, run["params"], p), protocol=5)).hexdigest()
    cache = execution.cache.setdefault("targeted_inputs", {})
    key = (declaration["label"], fingerprint)
    if key in cache:
        print("Targeted configuration already cached; no fit performed.")
        return cache[key]
    split_column, split_values = declaration["split_column"], declaration["split_values"]
    if execution.options["family"] == "descriptor":
        keys = list(execution.options["input_metadata"]["key_columns"])
        frame = baseline.to_pandas().copy()
        frame["eligible"] = np.isfinite(frame[list(batch.features)].to_numpy(dtype=float)).all(axis=1)
        frame["support_start"] = pd.to_datetime(frame.week_start) - pd.Timedelta(days=7)
        frame["support_end"] = pd.to_datetime(frame.week_start) + pd.Timedelta(days=13)
        if split_column is not None:
            frame = frame.merge(declaration["baseline_split"], on=keys, how="left", validate="one_to_one")
        result = targeted_descriptors(frame, declaration["treatments"], key_columns=keys,
            features=list(batch.features), week_column="week_start", split_column=split_column,
            split_values=split_values, settings=settings, seed_namespace=declaration["seed_namespace"])
    else:
        result = targeted_joint_paths({"paths":baseline, "split":declaration["baseline_split"]},
            declaration["treatments"], split_column=split_column, split_values=split_values,
            settings=settings, seed_namespace=declaration["seed_namespace"], include_dtw=method == "dtw",
            dtw_band=p.get("band", execution.options.get("dtw_band")),
            medoid_max_iter=run["params"].get("max_iter", execution.options.get("medoid_max_iter")))
    selected = {}
    for name, fitted in result["batches"].items():
        candidates = [r for r in fitted.runs.values() if r["k"] == run["k"] and
                      fitted.representations[r["representation_id"]]["params"]["method"] == method]
        selected[name] = candidates[0] if len(candidates) == 1 else None
    retained = {"execution_id":execution.id, "family":execution.options["family"], "fingerprint":fingerprint,
                "declaration":snapshot, "settings":settings, "result":result, "selected_runs":selected}
    cache[key] = retained
    return retained

In [ ]:
def matched_medoid_inspection(execution, config, method, k, restart=None):
    """Adapt a cached converged matched-medoid restart to common inspector inputs.

    Returns a context containing execution, batch, run, rep, rows, history,
    key_columns, focal_days, reference_days and preparation. No fit is performed.
    """
    from types import SimpleNamespace
    from market_analysis.behavior_paths import dependent_dtw, summarize_joint_paths
    from market_analysis.behavior_workbench import _id

    if execution.options.get("family") != "joint_paths":
        raise ValueError("Matched inspection requires a joint-path execution")
    retained = execution.cache.get("matched_medoids", {}).get(config)
    if retained is None or retained.get("execution_id") != execution.id or retained.get("config") != config:
        raise ValueError("Matched configuration is not retained for this execution")
    paths, result = retained["paths"], retained["result"]
    tensor = np.asarray(paths.tensor, dtype=float)
    transformed = np.asarray(paths.transformed, dtype=float)
    scales = np.asarray(paths.scales, dtype=float)
    keys = tuple(paths.keys.itertuples(index=False, name=None))
    provenance = {"keys": keys, "key_columns": tuple(paths.keys.columns),
                  "channels": tuple(paths.channels), "transforms": tuple(paths.transforms),
                  "scales": scales.tolist(), "subset_seed": paths.subset_seed}
    if (tensor.ndim != 3 or len(tensor) != len(keys) or len(set(keys)) != len(keys)
            or not np.isfinite(tensor).all() or scales.shape != (tensor.shape[2],)
            or not np.isfinite(scales).all() or np.any(scales <= 0)
            or transformed.shape != tensor.shape or not np.array_equal(transformed / scales, tensor)
            or result.get("provenance") != provenance
            or result.get("population_id") != _id({"tensor": tensor.tolist(), "provenance": provenance})):
        raise ValueError("Matched path keys, scales or tensor do not match retained result provenance")
    if method not in ("aligned", "dtw") or k not in result["groups"]:
        raise ValueError("Choose a retained matched method and cluster count")
    group = result["groups"][k][method]
    selected = group["best_restart"] if restart is None else restart
    records = [r for r in group["restarts"] if r["restart"] == selected]
    if len(records) != 1 or records[0].get("status") != "ok" or records[0].get("converged") is not True:
        raise ValueError("Matched inspection requires a successful converged restart")
    record = records[0]
    labels = np.asarray(record["assignments"])
    medoid_keys = tuple(record["medoid_keys"])
    if (labels.shape != (len(keys),) or not np.issubdtype(labels.dtype, np.integer)
            or np.any(labels < 0) or np.any(labels >= k) or len(medoid_keys) != k
            or len(set(medoid_keys)) != k or any(key not in keys for key in medoid_keys)):
        raise ValueError("Matched assignments or medoid keys do not align with paths")
    controls = retained["preparation_controls"]
    fingerprint = execution.preparation.get("source_fingerprint")
    if (not fingerprint or source_fingerprint(execution.source["history"]) != fingerprint
            or controls != execution.preparation["controls"]):
        raise ValueError("Matched preparation does not match retained source snapshots")
    identity = _id([execution.id, config, result["population_id"], result["dtw_band"], method, k, selected,
                    labels.tolist(), medoid_keys, record["seed"], record["iterations"], record["objective"]])
    cache = execution.cache.setdefault("matched-inspection-v1", {})
    if identity in cache:
        return cache[identity]
    indices = [keys.index(key) for key in medoid_keys]
    if method == "aligned":
        flat = tensor.reshape(len(keys), -1)
        distances = np.linalg.norm(flat[:, None, :] - flat[indices][None, :, :], axis=2)
    else:
        distances = np.array([[dependent_dtw(path, tensor[i], band=result["dtw_band"]) for i in indices]
                              for path in tensor])
    key_columns = list(paths.keys.columns)
    metadata = execution.source["history"].select("region_id", "type_id", "type_name", "study_category").unique().to_pandas()
    selected_rows = paths.keys.merge(metadata, on=["region_id", "type_id"], how="left", validate="many_to_one", sort=False)
    candidates = paths.candidates.copy()
    rows = selected_rows.merge(candidates[key_columns + ["price_reference", "activity_reference"]],
                               on=key_columns, how="left", validate="one_to_one", sort=False)
    rows = rows.merge(summarize_joint_paths(paths), on=key_columns, how="left", validate="one_to_one", sort=False)
    if len(rows) != len(keys) or rows["type_name"].isna().any() or not rows[key_columns].equals(paths.keys[key_columns]):
        raise ValueError("Missing or misaligned matched market metadata")
    rows["price_log_change"] = rows["observed_log_price_change"]
    rows["price_range"] = rows["observed_log_price_range"]
    rows["activity_ratio"] = rows["mean_activity_ratio"]
    rows["group"] = labels.copy()
    history = execution.source["history"].select("region_id", "type_id", "market_date", "vwap", "volume", "order_count", "is_observed").to_pandas()
    history["market_date"] = pd.to_datetime(history["market_date"])
    run_id, rep_id = "matched-run-" + identity, "matched-rep-" + identity
    rep = {"id": rep_id, "params": {"method": method, **({"band": result["dtw_band"]} if method == "dtw" else {})},
           "seed": None, "embedding": tensor.reshape(len(keys), -1).copy() if method == "aligned" else None}
    run = {"id": run_id, "representation_id": rep_id, "status": "ok", "reason": None,
           "keys": keys, "k": k, "seed": record["seed"], "assignments": labels.copy(),
           "centroid_distances": distances, "medoid_keys": medoid_keys,
           "medoid_paths": tensor[indices].copy(), "details": {"converged": record["converged"],
           "iterations": record["iterations"], "objective": record["objective"]},
           "params": {"method": "alternating_medoids", "n_clusters": k, "repetition": selected},
           "native": {}, "common_input": {}}
    batch = SimpleNamespace(keys=keys, prepared=tensor.reshape(len(keys), -1).copy(),
        population_id="matched-" + result["population_id"], features=tuple(paths.channels),
        sample_indices=np.arange(len(keys)), runs={run_id: run}, representations={rep_id: rep},
        input_metadata={"family": "joint_paths", "key_columns": tuple(key_columns),
                        "channels": tuple(paths.channels), "scales": scales.copy()})
    preparation = {"paths": paths, "candidates": candidates, "selected": selected_rows,
                   "controls": dict(controls), "source_fingerprint": fingerprint, "population_id": batch.population_id}
    context = {"execution": execution, "batch": batch, "family": "joint_paths", "run": run, "rep": rep,
               "rows": rows, "history": history, "key_columns": key_columns, "preparation": preparation,
               "focal_days": controls["window"], "reference_days": max(14, controls["reference_days"])}
    cache[identity] = context
    return context

In [ ]:
def plot_shortlist_seed_check(batch, run_ids):
    from sklearn.metrics import adjusted_rand_score
    fig = go.Figure()
    if len(set(run_ids)) != len(run_ids):
        raise ValueError("Shortlist run IDs must be distinct")
    if any(run_id not in batch.runs for run_id in run_ids):
        raise ValueError("Unknown or stale shortlist run ID")
    runs = [batch.runs[run_id] for run_id in run_ids]
    if any(r["status"] != "ok" for r in runs):
        raise ValueError("Shortlist must contain successful runs")
    def comparable(run):
        rep = batch.representations[run["representation_id"]]
        representation = {k: v for k, v in rep["params"].items() if k != "repetition"}
        clustering = {k: v for k, v in run["params"].items() if k != "repetition"}
        return representation, clustering
    if runs:
        if any(tuple(r["keys"]) != tuple(batch.keys) for r in runs):
            raise ValueError("Shortlist keys must exactly match the selected batch")
        if any(comparable(r) != comparable(runs[0]) for r in runs[1:]):
            raise ValueError("Shortlist configurations differ beyond repetition")
        repetitions = [batch.representations[r["representation_id"]]["params"].get("repetition", r["params"].get("repetition", 0)) for r in runs]
        if len(set(repetitions)) != len(repetitions):
            raise ValueError("Select distinct repetitions of the same configuration")
    if len(runs) < 2:
        fig.add_annotation(text="Not computed: select at least two matching fitted repetitions", x=.5, y=.5, xref="paper", yref="paper", showarrow=False)
    else:
        labels = [r["id"][:12] for r in runs]
        z = np.empty((len(runs), len(runs)))
        hover = np.empty_like(z, dtype=object)
        for i, a in enumerate(runs):
            for j, b in enumerate(runs):
                ari = adjusted_rand_score(a["assignments"], b["assignments"])
                agreement = np.mean(np.asarray(a["assignments"]) == np.asarray(b["assignments"]))
                z[i, j] = ari
                hover[i, j] = f"ARI: {ari:.3f}<br>raw label agreement: {agreement:.1%} (label-sensitive)<br>{a['id']} · seed {a['seed']}<br>{b['id']} · seed {b['seed']}"
        fig.add_trace(go.Heatmap(z=z, x=labels, y=labels, zmin=-1, zmax=1, colorscale="Viridis", text=hover, hovertemplate="%{text}<extra></extra>", colorbar_title="ARI"))
    fig.update_layout(title=f"Seed check · population {batch.population_id[:12]} · supporting diagnostic only", xaxis_title="Run", yaxis_title="Run")
    return fig

In [ ]:
def plot_matched_medoid_comparison(retained, selected_k, selected_restart=None):
    """Read-only figure; objectives use separate method-local axes."""
    result = retained["result"]
    if selected_k not in result["groups"]:
        raise ValueError("Display K must be in the computed configuration")
    if selected_restart is not None and (not isinstance(selected_restart, int) or
                                          not 0 <= selected_restart < result["starts"]):
        raise ValueError("Select a restart index in the computed range, or None")
    methods = ("aligned", "dtw")
    selected = {}
    subtitles = []
    for method in methods:
        group = result["groups"][selected_k][method]
        index = group["best_restart"] if selected_restart is None else selected_restart
        record = next((r for r in group["restarts"] if r["restart"] == index), None)
        selected[method] = record
        if record is None:
            status = "optimization-incomplete: no converged restart"
        elif record["status"] != "ok":
            status = f"restart {index}: failed"
        else:
            status = f"restart {index}: " + ("converged" if record["converged"] else "NONCONVERGED")
        subtitles.append(f"{method} · K={selected_k} · {status}")
    fig = make_subplots(rows=2, cols=2, subplot_titles=("Aligned · method-local objective",
        "Dependent DTW · method-local objective", *subtitles), vertical_spacing=.18)
    for col, method in enumerate(methods, 1):
        distance = result["distances"][method]
        counts = {"converged": 0, "nonconverged": 0, "failed": 0}
        details = [f"distance: {distance['status']} ({distance['fit_seconds']:.3f}s)" +
                   (f" · {distance['reason']}" if distance.get("reason") else "")]
        incomplete = []
        restart_seconds = 0.0
        for k in result["clusters"]:
            group = result["groups"][k][method]
            if group["status"] == "optimization-incomplete":
                incomplete.append(str(k))
            for record in group["restarts"]:
                status = ("converged" if record["converged"] else "nonconverged") if record["status"] == "ok" else "failed"
                counts[status] += 1
                restart_seconds += record["fit_seconds"]
                detail = f"K={k} restart={record['restart']}: {status}; {record['fit_seconds']:.3f}s"
                if record.get("reason"):
                    detail += f"; {record['reason']}"
                details.append(detail)
                if record["objective"] is not None:
                    fig.add_trace(go.Scatter(x=[f"K={k} · r={record['restart']}"], y=[record["objective"]],
                        mode="markers", marker=dict(symbol="circle" if record["converged"] else "x", size=10),
                        text=[detail], hovertemplate="%{text}<br>objective=%{y}<extra></extra>",
                        showlegend=False), row=1, col=col)
        summary = (f"{distance['status']} distance {distance['fit_seconds']:.3f}s · restarts {restart_seconds:.3f}s<br>"
                   f"{counts['converged']} converged · {counts['nonconverged']} nonconverged · {counts['failed']} failed")
        if incomplete:
            summary += " · optimization-incomplete K=" + ",".join(incomplete)
        fig.add_annotation(text=summary, hovertext="<br>".join(details),
                           x=.22 if col == 1 else .78, y=1.08, xref="paper", yref="paper",
                           xanchor="center", yanchor="bottom", showarrow=False)
        record = selected[method]
        if record is not None and record["status"] == "ok":
            labels, group_counts = np.unique(record["assignments"], return_counts=True)
            fig.add_trace(go.Bar(x=[str(x) for x in labels], y=group_counts,
                hovertemplate="group %{x}<br>count %{y}<extra></extra>", showlegend=False), row=2, col=col)
        fig.update_yaxes(title_text="Objective (within method only)", row=1, col=col)
        fig.update_yaxes(title_text="Assigned paths", row=2, col=col)
        fig.update_xaxes(title_text="Run-specific group", row=2, col=col)
    fig.update_layout(title=f"Matched medoids · {len(result['provenance']['keys']):,} paths · "
                      f"population {result['population_id'][:12]} · objectives not comparable across distances",
                      height=800, margin=dict(t=170))
    return fig

In [ ]:
def plot_calendar_shortlist(retained, feature):
    result, chosen = retained["result"], retained["chosen"]
    mode = result["mode"]
    if feature is None:
        feature = retained["features"][0] if retained["family"] == "descriptor" else "observed_log_price_range"
    if any(feature not in frame.columns for frame in result["profiles"].values()):
        raise ValueError(f"Original-unit profile feature unavailable: {feature}")
    fig = make_subplots(rows=2, cols=2, subplot_titles=("Coverage", "Assignment distance" if mode == "transfer" else "Independent rediscovery",
        "Training groups", "Heldout groups"), vertical_spacing=.18)
    coverage = result["coverage"]
    counts = [coverage[name]["eligible"] if isinstance(coverage[name], dict) else coverage[name] for name in ("training", "heldout")]
    fig.add_trace(go.Bar(x=["training", "heldout", "outside periods"], y=[*counts, coverage["outside_periods"]],
                         name="eligible selected", showlegend=False), row=1, col=1)
    if isinstance(coverage["training"], dict):
        fig.add_trace(go.Bar(x=["training", "heldout"], y=[coverage[n]["ineligible"] for n in ("training", "heldout")],
                             name="ineligible"), row=1, col=1)
    train_run = result["batches"]["training"].runs[chosen["training"]]
    labels = {"training": np.asarray(train_run["assignments"])}
    train_dist = train_run["centroid_distances"][np.arange(len(labels["training"])), labels["training"]]
    if mode == "transfer":
        inference = result["transfer"][chosen["training"]]
        labels["heldout"] = np.asarray(inference["assignments"])
        for period, distances in (("training", train_dist), ("heldout", inference["assignment_distances"])):
            fig.add_trace(go.Box(y=distances, name=period, boxpoints=False, showlegend=False), row=1, col=2)
    else:
        held_run = result["batches"]["heldout"].runs[chosen["heldout"]]
        labels["heldout"] = np.asarray(held_run["assignments"])
        fig.add_annotation(text="Independent fits: labels are not matched.<br>Groups may split, merge or disappear.",
                           x=.78, y=.78, xref="paper", yref="paper", showarrow=False)
    for col, period in enumerate(("training", "heldout"), 1):
        frame = result["profiles"][period]
        if len(frame) != len(labels[period]):
            raise ValueError("Profile and assignment populations do not align")
        for group in np.unique(labels[period]):
            values = pd.to_numeric(frame.loc[labels[period] == group, feature], errors="coerce").dropna()
            fig.add_trace(go.Box(y=values, name=f"{period} · group {group}", boxpoints=False, showlegend=False), row=2, col=col)
        fig.update_yaxes(title_text=feature + " · original units", row=2, col=col)
    fig.update_layout(title=f"Calendar {mode} · execution {retained['execution_id'][:12]} · descriptive, not accuracy",
                      height=780, barmode="stack")
    return fig

In [ ]:
def plot_window_origin(retained, features, profile_feature, pair_index):
    """Compare overlapping windows and show one source-observation context."""
    left, right, matches = retained["original_profiles"], retained["shifted_profiles"], retained["matches"]
    keys, markets = list(retained["key_columns"]), list(retained["market_columns"])
    candidates = list(features) if retained["family"] == "descriptor" else ["median_log_price_relative", "final_log_price_relative", "observed_log_price_change", "observed_log_price_range", "price_peak_timing", "mean_activity_ratio", "peak_activity_share", "activity_peak_timing", "observed_price_coverage"]
    feature = profile_feature if profile_feature is not None else next((x for x in candidates if x in left and x in right), None)
    if feature not in left or feature not in right:
        raise ValueError("Select an available original-unit profile feature")
    if not isinstance(pair_index, int) or isinstance(pair_index, bool) or pair_index < 0:
        raise ValueError("Matched pair index must be a non-negative integer")
    activity = "volume" if retained["family"] == "descriptor" else retained["original_paths"].channels[1]
    fig = make_subplots(rows=3, cols=2, specs=[[{}, {}], [{}, {"secondary_y": True}], [{"colspan": 2}, None]],
                        subplot_titles=("Matched-window coverage", "Overlap and phase", f"Paired profile difference · {feature}", "Selected pair · observed price and activity", "Membership transitions · overlap pairs (not independent events)"))
    def side_keys(prefix):
        return [c if c in markets else f"{prefix}_{c}" for c in keys]
    left_unique = len(matches[side_keys("left")].drop_duplicates()) if len(matches) else 0
    right_unique = len(matches[side_keys("right")].drop_duplicates()) if len(matches) else 0
    fig.add_trace(go.Bar(x=["original", "shifted"], y=[left_unique, right_unique], name="matched", marker_color="#3a8d76"), row=1, col=1)
    fig.add_trace(go.Bar(x=["original", "shifted"], y=[len(left)-left_unique, len(right)-right_unique], name="unmatched", marker_color="#b4b4b4"), row=1, col=1)
    fig.update_layout(barmode="stack")
    fig.update_yaxes(title_text="Eligible windows", row=1, col=1)
    if len(matches):
        fig.add_trace(go.Scatter(x=matches.left_overlap_start_offset, y=matches.right_overlap_start_offset,
            mode="markers", marker=dict(size=7, color=matches.overlap_days, colorscale="Viridis", showscale=True, colorbar=dict(title="Overlap days", x=1.02)),
            customdata=np.column_stack((matches.overlap_days, matches.left_episode_start.astype(str), matches.right_episode_start.astype(str))),
            hovertemplate="original offset %{x}<br>shifted offset %{y}<br>overlap %{customdata[0]} days<br>original %{customdata[1]}<br>shifted %{customdata[2]}<extra></extra>", showlegend=False), row=1, col=2)
        for prefix, assignments in (("left", retained["original_assignments"]), ("right", retained["shifted_assignments"])):
            labels = assignments[keys + ["group"]].rename(columns={k: k if k in markets else f"{prefix}_{k}" for k in keys}).rename(columns={"group": f"{prefix}_group"})
            matches = matches.merge(labels, on=side_keys(prefix), how="left", validate="many_to_one", sort=False)
        if matches[["left_group", "right_group"]].isna().any().any():
            raise ValueError("Matched windows are missing keyed assignments")
        transitions = pd.crosstab(matches.left_group, matches.right_group)
        fig.add_trace(go.Heatmap(x=[str(x) for x in transitions.columns], y=[str(x) for x in transitions.index],
            z=transitions.to_numpy(), colorscale="Blues", showscale=False,
            hovertemplate="original group %{y}<br>shifted group %{x}<br>overlap pairs %{z}<extra></extra>"), row=3, col=1)
        def keyed_values(frame, prefix):
            renamed = frame[keys + [feature]].rename(columns={k: k if k in markets else f"{prefix}_{k}" for k in keys})
            return renamed.rename(columns={feature: prefix + "_value"})
        pairs = matches.merge(keyed_values(left, "left"), on=side_keys("left"), validate="many_to_one")
        pairs = pairs.merge(keyed_values(right, "right"), on=side_keys("right"), validate="many_to_one")
        fig.add_trace(go.Histogram(x=pairs.right_value - pairs.left_value, name="shifted − original", showlegend=False), row=2, col=1)
        if pair_index >= len(matches):
            raise ValueError(f"Matched pair index must be below {len(matches)}")
        pair = matches.iloc[pair_index]
        def find_profile(frame, prefix):
            mask = np.ones(len(frame), dtype=bool)
            for key in keys:
                value = pair[key if key in markets else f"{prefix}_{key}"]
                series = frame[key]
                if key in ("week_start", "start"):
                    mask &= (pd.to_datetime(series) == pd.Timestamp(value)).to_numpy()
                else:
                    mask &= (series == value).to_numpy()
            return frame.loc[mask].iloc[0]
        a, b = find_profile(left, "left"), find_profile(right, "right")
        history = retained["source_history"].to_pandas()
        mask = np.ones(len(history), dtype=bool)
        for key in markets:
            mask &= (history[key] == pair[key]).to_numpy()
        date = pd.to_datetime(history["market_date"])
        mask &= ((date >= min(a.support_start, b.support_start)) & (date <= max(a.support_end, b.support_end))).to_numpy()
        history = history.loc[mask].sort_values("market_date").copy()
        if "is_observed" in history:
            history.loc[~history["is_observed"].fillna(False), "vwap"] = np.nan
        fig.add_trace(go.Scatter(x=history.market_date, y=history.vwap, mode="lines+markers", name="observed VWAP (raw)",
                                 line=dict(color="#7042a0"), connectgaps=False), row=2, col=2, secondary_y=False)
        fig.add_trace(go.Scatter(x=history.market_date, y=history[activity], mode="lines", name=f"{activity} (raw activity)",
                                 line=dict(color="#7042a0", dash="dash"), connectgaps=False), row=2, col=2, secondary_y=True)
        for label, record, color in (("original", a, "#3366aa"), ("shifted", b, "#dd8844")):
            fig.add_vrect(x0=record.episode_start, x1=record.episode_end + pd.Timedelta(days=1), fillcolor=color, opacity=.15, line_width=0, row=2, col=2)
        market_label = ", ".join(f"{key}={pair[key]}" for key in markets)
        fig.layout.annotations[3].update(text=(f"Selected pair {pair_index} · {market_label}<br>"
            f"original {a.episode_start:%Y-%m-%d} (blue), group {pair.left_group}<br>"
            f"shifted {b.episode_start:%Y-%m-%d} (orange), group {pair.right_group}"))
    else:
        fig.add_annotation(text="No qualifying same-market overlaps", x=.5, y=.5, xref="paper", yref="paper", showarrow=False)
    fig.update_xaxes(title_text="Original overlap offset (days)", row=1, col=2)
    fig.update_yaxes(title_text="Shifted overlap offset (days)", row=1, col=2)
    fig.update_xaxes(title_text="Shifted − original (feature units)", row=2, col=1)
    fig.update_yaxes(title_text="Matched pairs", row=2, col=1)
    fig.update_yaxes(title_text="Observed VWAP (raw price)", row=2, col=2, secondary_y=False)
    fig.update_yaxes(title_text=f"{activity} (raw activity)", row=2, col=2, secondary_y=True)
    note = " · following-week context is fitted, not independent validation" if retained["family"] == "descriptor" else ""
    fig.update_xaxes(title_text="Shifted group · same fitted pipeline", row=3, col=1)
    fig.update_yaxes(title_text="Original group", row=3, col=1)
    fig.update_layout(title=f"Window origin · {len(matches)} overlap pairs (windows may repeat) · shift {retained['shift_days']} days{note}", height=1100, margin=dict(b=100))
    return fig

In [ ]:
def plot_targeted_inputs(retained, profile_feature):
    """Show common-population evidence without comparing raw labels across runs."""
    result = retained["result"]
    profiles = result["frames"] if retained["family"] == "descriptor" else result["profiles"]
    common_columns = set.intersection(*(set(frame.columns) for frame in profiles.values()))
    candidates = (list(result["batches"]["baseline"].features) if retained["family"] == "descriptor" else
                  ["median_log_price_relative", "observed_log_price_range", "mean_activity_ratio", "peak_activity_share"])
    feature = profile_feature if profile_feature is not None else next((x for x in candidates if x in common_columns), None)
    if feature not in common_columns:
        raise ValueError("Select a profile feature available in every treatment")
    fig = make_subplots(rows=2, cols=2, subplot_titles=("Eligibility and common retention", f"Original-unit profile · {feature}", "Training feature by run-specific group", "Paired seed and fit status"))
    retention = result["retention"]
    fig.add_trace(go.Bar(x=retention.treatment, y=retention.eligible, name="eligible"), row=1, col=1)
    fig.add_trace(go.Bar(x=retention.treatment, y=retention.common, name="common"), row=1, col=1)
    for name, frame in profiles.items():
        fig.add_trace(go.Box(y=frame[feature], name=name, boxpoints=False, showlegend=False), row=1, col=2)
        run = retained["selected_runs"][name]
        if run is not None and run["status"] == "ok":
            keys = list(result["keys"].columns)
            assignments = pd.DataFrame(list(run["keys"]), columns=keys)
            assignments["group"] = np.asarray(run["assignments"])
            training = result["keys"].loc[np.asarray(result["training_mask"], dtype=bool), keys]
            aligned = training.merge(frame[keys + [feature]], on=keys, how="left", validate="one_to_one", sort=False)
            aligned = aligned.merge(assignments, on=keys, how="left", validate="one_to_one", sort=False)
            if aligned["group"].isna().any() or len(assignments) != len(training):
                raise ValueError(f"{name}: training assignments do not match common keys")
            for group, values in aligned.groupby("group", sort=True):
                fig.add_trace(go.Box(y=values[feature], name=f"{name} · group {group} (n={len(values)})", boxpoints=False, showlegend=False), row=2, col=1)
    lines = []
    for name, run in retained["selected_runs"].items():
        if run is None:
            lines.append(f"{name}: requested run missing or ambiguous")
        elif run["status"] != "ok":
            lines.append(f"{name}: {run['status']} · {run.get('reason')}")
        else:
            batch = result["batches"][name]
            rep = batch.representations[run["representation_id"]]
            lines.append(f"{name}: ok · cluster seed {run['seed']} · representation seed {rep['seed']}")
    fig.add_annotation(text="<br>".join(lines), x=.77, y=.18, xref="paper", yref="paper", showarrow=False, align="left")
    fig.update_yaxes(title_text="Windows", row=1, col=1)
    fig.update_yaxes(title_text=f"{feature} · original units", row=1, col=2)
    fig.update_yaxes(title_text=f"{feature} · training keys · original units", row=2, col=1)
    split = "full common population" if retained["declaration"]["split_column"] is None else "explicit training / heldout split"
    fig.update_layout(title=f"Targeted inputs · {split} · namespace {result['seed_namespace']} · {retained['fingerprint'][:12]}<br>Run-specific groups; labels are not equivalent across refits", height=800)
    return fig

In [ ]:
def shortlist_mode_args(mode):
    """Read only the selected mode's controls; never calculate."""
    if mode == "seed":
        return (shortlist_run_ids,)
    if mode == "matched_medoids":
        return (matched_subset_size, matched_clusters, matched_dtw_band, matched_max_iter,
                matched_starts, matched_seed, matched_display_k, matched_display_restart)
    if mode in ("calendar_transfer", "calendar_rediscovery"):
        return (calendar_run_id, calendar_training_period, calendar_heldout_period, calendar_profile_feature)
    if mode == "window_origin":
        return (origin_run_id, origin_shift_days, origin_min_overlap_days, origin_profile_feature, origin_pair_index)
    if mode == "targeted_inputs":
        return (target_run_id, target_experiment_label, target_treatments, target_split_column,
                target_split_values, target_baseline_split, target_seed_namespace, target_profile_feature)
    raise ValueError("Choose a supported shortlist mode")

In [ ]:
def plot_shortlist_board(execution, mode, *args):
    """Select only the active execution's cached result; never prepare or fit."""
    if mode == "targeted_inputs":
        target_id, target_label, treatments, split_column, split_values, baseline_split, seed_namespace, target_feature = args
        try:
            declaration = targeted_controls(execution, target_id, target_label, treatments, split_column, split_values, baseline_split, seed_namespace)
            candidates = [item for (label, fingerprint), item in execution.cache.get("targeted_inputs", {}).items()
                          if label == target_label and targeted_equal(item["declaration"], declaration)]
            if not candidates:
                raise ValueError("No cached result for the current execution and declarations; calculate explicitly first")
            return plot_targeted_inputs(candidates[-1], target_feature)
        except (ValueError, TypeError, KeyError) as exc:
            fig = go.Figure()
            fig.add_annotation(text=f"Targeted inputs not computed: {exc}", x=.5, y=.5, xref="paper", yref="paper", showarrow=False)
            return fig
    if mode == "window_origin":
        origin_id, origin_shift, origin_overlap, origin_feature, origin_pair = args
        try:
            config = window_origin_config(execution, origin_id, origin_shift, origin_overlap)
            retained = execution.cache.get("window_origin", {}).get(config)
            if retained is None:
                raise ValueError("No cached window-origin result for the active execution and controls; calculate explicitly first")
            return plot_window_origin(retained, execution.batch.features, origin_feature, origin_pair)
        except ValueError as exc:
            fig = go.Figure()
            fig.add_annotation(text=f"Window origin not computed: {exc}", x=.5, y=.5, xref="paper", yref="paper", showarrow=False)
            return fig
    if mode == "seed":
        run_ids, = args
        return plot_shortlist_seed_check(execution.batch, run_ids)
    if mode in ("calendar_transfer", "calendar_rediscovery"):
        calendar_id, training_period, heldout_period, profile_feature = args
        try:
            config = calendar_shortlist_config(execution, mode, calendar_id, training_period, heldout_period)
            retained = execution.cache.get("calendar_checks", {}).get(config)
            if retained is None:
                raise ValueError("No cached calendar result for the active execution and controls; calculate explicitly first")
            return plot_calendar_shortlist(retained, profile_feature)
        except ValueError as exc:
            fig = go.Figure()
            fig.add_annotation(text=str(exc), x=.5, y=.5, xref="paper", yref="paper", showarrow=False)
            return fig
    if mode != "matched_medoids":
        raise ValueError("Choose seed, matched_medoids, calendar_transfer, calendar_rediscovery, window_origin or targeted_inputs")
    subset_size, clusters, band, max_iter, starts, seed, selected_k, selected_restart = args
    config = matched_medoid_config(subset_size, clusters, band, max_iter, starts, seed)
    if execution.options.get("family") != "joint_paths":
        raise ValueError("Matched medoids require a retained joint-path execution")
    retained = execution.cache.get("matched_medoids", {}).get(config)
    if retained is None:
        fig = go.Figure()
        fig.add_annotation(text="No cached matched comparison for the active execution and controls; calculate explicitly first.",
                           x=.5, y=.5, xref="paper", yref="paper", showarrow=False)
        return fig
    return plot_matched_medoid_comparison(retained, selected_k, selected_restart)

In [ ]:
def calculate_selected_shortlist(mode):
    """Plan and optionally calculate only the selected check."""
    # Calculation only: plan first; fitting requires the explicit toggle.
    if mode not in ("seed", "matched_medoids", "calendar_transfer", "calendar_rediscovery", "window_origin", "targeted_inputs"):
        raise ValueError("Choose a supported shortlist mode")
    if mode == "targeted_inputs":
        _shortlist_execution = require_active_execution()
        try:
            _target_declaration = targeted_controls(_shortlist_execution, target_run_id, target_experiment_label,
                target_treatments, target_split_column, target_split_values, target_baseline_split, target_seed_namespace)
            print("Targeted plan: " + ("full common population" if target_split_column is None else "explicit train/heldout split") +
                  f"; baseline plus {len(target_treatments)} treatment(s), selected run configuration only.")
            if target_calculate:
                calculate_targeted_inputs(_shortlist_execution, _target_declaration)
        except (ValueError, TypeError, KeyError) as exc:
            print(f"Targeted inputs not computed: {exc}")
    if mode == "window_origin":
        _shortlist_execution = require_active_execution()
        try:
            _origin_config = window_origin_config(_shortlist_execution, origin_run_id, origin_shift_days, origin_min_overlap_days)
            print("Window-origin plan: apply the selected fixed pipeline to shifted eligible windows; no fitting.")
            if origin_calculate:
                calculate_window_origin(_shortlist_execution, *_origin_config)
        except ValueError as exc:
            print(f"Window origin not computed: {exc}")
    if mode in ("calendar_transfer", "calendar_rediscovery"):
        _shortlist_execution = require_active_execution()
        try:
            _calendar_config = calendar_shortlist_config(_shortlist_execution, mode, calendar_run_id,
                                                          calendar_training_period, calendar_heldout_period)
            print("Calendar plan: selected run configuration, one fit per requested period; "
                  "dependent-DTW also fits the aligned control. Work and memory require calibration.")
            if calendar_calculate:
                calculate_calendar_shortlist(_shortlist_execution, _calendar_config)
        except ValueError as exc:
            print(f"Calendar not computed: {exc}")
    if mode == "matched_medoids":
        _shortlist_execution = require_active_execution()
        _matched_config = matched_medoid_config(matched_subset_size, matched_clusters, matched_dtw_band,
                                                 matched_max_iter, matched_starts, matched_seed)
        _matched_plan = matched_medoid_plan(_shortlist_execution, _matched_config)
        print(f"Matched plan: {_matched_plan['selected']:,} / {_matched_plan['eligible']:,} eligible paths; "
              f"{_matched_plan['distance_matrices']} distance matrices, {_matched_plan['restart_attempts']} restart attempts; "
              f"each dense matrix approximately {_matched_plan['pairwise_bytes_each'] / 2**20:.1f} MiB. "
              "Excludes temporary allocations and preparation; counts do not establish affordability.")
        if matched_calculate:
            calculate_matched_medoids(_shortlist_execution, _matched_config)

### Optional shortlist board

Choose a mode in the controls. For matched medoids, select a retained joint-path execution and run the calculation cell to review the plan. Set `matched_calculate` to `True` and rerun that cell only when an explicit calculation is wanted. Results are cached by execution and configuration, including every restart, assignment and initial key. The separate figure cell only reads the active execution and current controls, even if `matched_calculate` remains `True`. Hover over the compact status summary for individual restart details. To inspect a converged restart in the common boards, run the optional controls cell to display retained configuration numbers, then use the matched selection example in section 04. The section 05 definition cell must have been run first. For calendar checks, supply a successful run ID and explicit inclusive periods, then set `calendar_calculate` to `True` and rerun only the calculation cell. The figure reads cached results only; changing `calendar_profile_feature` never fits. Joint coverage refers to the retained selected eligible paths, not rejected candidates. For window origin, enter a successful run ID, an explicit integer shift and a positive minimum overlap, then set `origin_calculate` to `True` and run only the calculation cell. The figure cell reads the current execution and cached configuration; profile and pair controls do not calculate. For targeted inputs, supply a run ID, experiment label, seed namespace and explicit treatment declarations matching the helper interfaces; optionally supply a keyed baseline split and explicit split column/values. None/None uses the full common population. Descriptor treatments map names to `{"frame": pandas_frame, "features": (...), "changed_features": (...)}`; joint treatments map names to `{"paths": joint_paths, "split": keyed_frame_or_None, "channels": (...), "changed_channels": (...)}`. Frames must include the support, eligibility and split fields required by the corresponding helper. Set `target_calculate` only for explicit fitting; the figure checks declarations against the cached snapshot without hashing or fitting.

In [ ]:
calculate_selected_shortlist(shortlist_mode)

In [ ]:
plot_shortlist_board(require_active_execution(), shortlist_mode, *shortlist_mode_args(shortlist_mode)) if shortlist_show else None

### Optional input, neighborhood, or AE reconstruction board

Edit optional choices in the same per-execution `inspection_state` below. Input mode switches between raw and prepared descriptor coordinates on the identical fitted rows, or shows joint eligibility, interpolation burden, observed summaries, and an observed-versus-prepared selected path. Raw coordinate units differ by feature; their spreads are not standardized importance scores. Geometry mode compares query and neighbor histories alongside aligned prepared-input distance contributions; joint distances are not DTW contributions. Reconstruction mode is available for a selected descriptor AE representation and reports retained full and validation feature errors in squared prepared-coordinate units. AE history mode shows retained validation loss, the best epoch used for the full-data refit, and the stopping outcome; training losses are not retained. Descriptor neighbor histories include the prior, focal, and following weeks, with the focal week shaded. No mode refits a model.

In [ ]:
# Optional controls share the active execution's inspection state.
inspection_state = inspection_state_for(require_active_execution())
# inspection_state["optional_mode"] = "input"  # geometry, reconstruction, ae_history
# inspection_state["optional_query_key"] = None  # exact fitted key overrides row
# inspection_state["optional_query_row"] = 0
# inspection_state["optional_neighbor_rank"] = 1
# inspection_state["optional_coordinate"] = 0  # zero-based prepared coordinate
# inspection_state["optional_neighbors_shown"] = 20
# inspection_state["inspection_matched_page"] = 0  # zero-based matched configuration page

inspection_matched_page(require_active_execution(), inspection_state["inspection_matched_page"])

In [ ]:
def plot_optional_input_geometry(batch, rows, retained, history, rep, mode, query_key, query_row, neighbor_rank, coordinate, neighbors_shown):
    """Plot retained preparation, common-input neighbors, or descriptor AE errors."""
    x = np.asarray(batch.prepared)
    if x.ndim != 2 or len(x) != len(batch.keys) or len(rows) != len(batch.keys):
        raise ValueError("Prepared input and keys must align")
    family = batch.input_metadata.get("family")
    if family == "descriptor":
        names = list(batch.features)
    elif family == "joint_paths":
        paths = retained["paths"]
        names = [f"day {day + 1} · {channel}" for day in range(paths.tensor.shape[1]) for channel in paths.channels]
    else:
        raise ValueError("Unsupported family")
    if len(names) != x.shape[1]:
        raise ValueError("Prepared coordinate names do not match")
    if mode == "ae_history":
        if family != "descriptor" or rep["params"]["method"] != "ae":
            raise ValueError("AE history is available only for descriptor AE")
        details, params = rep.get("details", {}), rep["params"]
        losses = details.get("validation_loss")
        if losses is None or len(losses) == 0:
            raise ValueError("Selected AE representation has no retained validation history")
        best = details["best_epoch"]
        budget, patience = params["max_epochs"], params["patience"]
        stale = len(losses) - best
        if len(losses) == budget and stale >= patience:
            termination = "early stopping coincided with epoch-budget exhaustion"
        elif len(losses) == budget:
            termination = "epoch budget exhausted"
        elif stale >= patience:
            termination = "early stopping"
        else:
            termination = "termination unknown from retained history"
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=list(range(1, len(losses) + 1)), y=losses, mode="lines+markers", name="validation MSE"))
        fig.add_trace(go.Scatter(x=[best], y=[losses[best - 1]], mode="markers", name="best epoch / full-data refit length", marker=dict(size=12, symbol="star")))
        fig.update_layout(title=f"AE validation history · {termination} · {len(losses)}/{budget} epochs · patience {patience}",
                          xaxis_title="Epoch", yaxis_title="Validation MSE · prepared coordinates", height=700)
        return fig
    if mode == "reconstruction":
        if family != "descriptor" or rep["params"]["method"] != "ae":
            raise ValueError("Reconstruction is available only for descriptor AE")
        summaries = rep.get("details", {}).get("reconstruction")
        if not summaries:
            raise ValueError("Selected AE representation has no retained reconstruction diagnostics")
        fig = go.Figure()
        for name in ("full", "validation"):
            summary = summaries[name]
            fig.add_trace(go.Bar(x=summary["features"], y=summary["mse_by_feature"], name=f"{name} (n={summary['count']})",
                hovertemplate="%{x}<br>MSE %{y:.6g}<extra>%{fullData.name}</extra>"))
        fig.update_layout(title="AE reconstruction · per-feature mean squared error", barmode="group",
                          yaxis_title="MSE · squared prepared-coordinate units", height=700)
        return fig
    if mode not in ("input", "geometry"):
        raise ValueError("Choose input, geometry, reconstruction, or ae_history")
    if query_key is None:
        q = query_row
    else:
        cols = ["type_id", "week_start"] if family == "descriptor" else list(batch.input_metadata["key_columns"])
        def normalize(key):
            return tuple(pd.Timestamp(v) if c in ("start", "week_start") else v for c, v in zip(cols, key, strict=True))
        q = [normalize(key) for key in batch.keys].index(normalize(query_key))
    if not 0 <= q < len(x):
        raise ValueError("Invalid query row")
    if mode == "input" and family == "descriptor":
        raw = rows.loc[:, list(batch.features)].to_numpy(dtype=float)
        if raw.shape != x.shape or not np.isfinite(raw).all():
            raise ValueError("Raw diagnostics must use the same finite fitted rows as prepared inputs")
        fig = make_subplots(rows=2, cols=2, subplot_titles=(
            "Candidate / eligible / fitted", "Coordinate spread", "Fitted-row correlations",
            f"Distribution · {names[coordinate]}"))
        fig.add_trace(go.Bar(x=["candidates", "eligible", "fitted"],
            y=[len(batch.eligible), int(np.asarray(batch.eligible).sum()), len(x)], showlegend=False), row=1, col=1)
        trace_groups = []
        for space, values in (("prepared", x), ("raw", raw)):
            first = len(fig.data)
            for i, name in enumerate(names):
                q1, median, q3 = np.quantile(values[:, i], [.25, .5, .75])
                fig.add_trace(go.Box(q1=[q1], median=[median], q3=[q3], name=name,
                    boxpoints=False, showlegend=False, visible=space == "prepared"), row=1, col=2)
            with np.errstate(divide="ignore", invalid="ignore"):
                corr = np.atleast_2d(np.corrcoef(values, rowvar=False))
            fig.add_trace(go.Heatmap(z=corr, x=names, y=names, zmin=-1, zmax=1,
                colorscale="RdBu", colorbar_title="correlation", visible=space == "prepared"), row=2, col=1)
            fig.add_trace(go.Histogram(x=values[:, coordinate], name=names[coordinate],
                showlegend=False, visible=space == "prepared"), row=2, col=2)
            trace_groups.append((space, range(first, len(fig.data))))
        buttons = []
        for space, indices in trace_groups:
            visible = [False] * len(fig.data)
            visible[0] = True
            for i in indices:
                visible[i] = True
            buttons.append(dict(label=space.title(), method="update", args=[
                {"visible": visible}, {"title.text": f"{space.title()} descriptors · same {len(x):,} fitted rows · population {batch.population_id[:12]}"}]))
        fig.update_layout(title=f"Prepared descriptors · same {len(x):,} fitted rows · population {batch.population_id[:12]}",
            height=950, margin=dict(t=120),
            updatemenus=[dict(buttons=buttons, active=0, x=0, y=1.12, xanchor="left")])
        return fig
    if mode == "input":
        candidates = paths.candidates.copy()
        cols = list(batch.input_metadata["key_columns"])
        episode = paths.daily.loc[paths.daily.role.eq("episode")]
        burden = episode.groupby(cols, sort=False).interpolated.sum().rename("interpolated_days").reset_index()
        candidates = candidates.merge(burden, on=cols, how="left", validate="one_to_one")
        fig = make_subplots(rows=2, cols=2, specs=[[{}, {}], [{}, {"secondary_y": True}]],
            subplot_titles=("Candidate eligibility", "Interpolation burden vs activity reference", "Observed movement and activity", "Selected observed vs prepared episode"))
        fig.add_trace(go.Bar(x=["candidates", "eligible", "fitted"], y=[len(candidates), int(candidates.eligible.sum()), len(x)], showlegend=False), row=1, col=1)
        for eligible in (True, False):
            subset = candidates.loc[candidates.eligible == eligible]
            if len(subset) > 2000:
                # Retain low-reference and high-interpolation tails plus a deterministic spread.
                subset = pd.concat([subset.nsmallest(500, "activity_reference"),
                    subset.nlargest(500, "interpolated_days"),
                    subset.iloc[np.linspace(0, len(subset)-1, 1000, dtype=int)]]).drop_duplicates(cols)
            fig.add_trace(go.Scattergl(x=subset.interpolated_days, y=subset.activity_reference, mode="markers", name="eligible" if eligible else "rejected",
                text=[str(tuple(v)) for v in subset[cols].itertuples(index=False, name=None)],
                hovertemplate="interpolated days %{x}<br>reference daily activity %{y}<br>%{text}<extra>%{fullData.name}</extra>"), row=1, col=2)
        for feature, label in (("observed_log_price_change", "Price change<br>(log)"), ("median_log_price_relative", "Median displacement<br>(log)"), ("mean_activity_ratio", "Mean activity<br>/ reference")):
            values = pd.to_numeric(rows[feature], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            if len(values):
                q1, median, q3 = values.quantile([.25, .5, .75])
                fig.add_trace(go.Box(x=[label], q1=[q1], median=[median], q3=[q3], name=f"{label} (n={len(values)})", boxpoints=False), row=2, col=1)
        selected = episode.copy()
        for col, value in zip(cols, batch.keys[q], strict=True):
            selected = selected.loc[selected[col] == value]
        selected = selected.sort_values("date")
        row = rows.iloc[q]
        observed = pd.to_numeric(selected.observed_price, errors="coerce").to_numpy(dtype=float)
        prepared = pd.to_numeric(selected.prepared_price, errors="coerce").to_numpy(dtype=float)
        raw_activity = pd.to_numeric(selected.observed_activity, errors="coerce").to_numpy(dtype=float)
        days = (pd.to_datetime(selected.date) - pd.Timestamp(row.start)).dt.days
        price_ref, activity_ref = row.price_reference, row.activity_reference
        detail = [f"interpolated: {flag}<br>left: {left}<br>right: {right}" for flag, left, right in zip(selected.interpolated, selected.interpolation_left, selected.interpolation_right, strict=True)]
        fig.add_trace(go.Scatter(x=days, y=observed / price_ref * 100, mode="lines+markers", name="observed price", line_color="#1f77b4", connectgaps=False,
            customdata=observed, hovertemplate="day %{x}<br>index %{y}<br>raw price %{customdata}<extra>%{fullData.name}</extra>"), row=2, col=2, secondary_y=False)
        fig.add_trace(go.Scatter(x=days, y=prepared / price_ref * 100, mode="lines+markers", name="prepared price (interpolated)", line=dict(color="#1f77b4", dash="dash"),
            text=detail, customdata=prepared, hovertemplate="day %{x}<br>index %{y}<br>prepared price %{customdata}<br>%{text}<extra>%{fullData.name}</extra>"), row=2, col=2, secondary_y=False)
        fig.add_trace(go.Scatter(x=days, y=raw_activity / activity_ref, mode="lines", name="observed activity / reference", line_color="#ff7f0e", customdata=raw_activity,
            hovertemplate="day %{x}<br>ratio %{y}<br>raw activity %{customdata}<extra>%{fullData.name}</extra>"), row=2, col=2, secondary_y=True)
        fig.update_yaxes(title_text="Price index (reference = 100)", row=2, col=2, secondary_y=False)
        fig.update_yaxes(title_text="Activity / reference", row=2, col=2, secondary_y=True)
        fig.update_layout(title=f"Joint input provenance · selected {batch.keys[q]} · burden scatter capped at 2,000 per status", height=950)
        return fig
    if len(x) < 2 or not 1 <= neighbor_rank < len(x):
        raise ValueError("A query and another fitted row are required")
    squared = ((x - x[q]) ** 2).sum(axis=1)
    squared[q] = np.inf
    ordered = np.argsort(squared, kind="stable")
    neighbor = ordered[neighbor_rank - 1]
    shown = ordered[:min(neighbors_shown, len(x)-1)]
    contributions = (x[q] - x[neighbor]) ** 2
    order = np.argsort(contributions)[::-1]
    fig = make_subplots(rows=2, cols=2, specs=[[{}, {}], [{}, {"secondary_y": True}]],
        subplot_titles=("Nearest prepared-input distances", "Selected neighbor · squared contributions", "Cumulative squared distance", "Query and neighbor · observed relative histories"))
    fig.add_trace(go.Bar(x=np.arange(1, len(shown)+1), y=np.sqrt(squared[shown]), text=[str(batch.keys[i]) for i in shown], hovertemplate="rank %{x}<br>distance %{y:.4g}<br>%{text}<extra></extra>", showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=[names[i] for i in order], y=contributions[order], showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=np.arange(1, len(order)+1), y=np.cumsum(contributions[order]), mode="lines+markers", showlegend=False), row=2, col=1)
    for i, style, label in ((q, "solid", "query"), (neighbor, "dash", "neighbor")):
        row = rows.iloc[i]
        start = pd.Timestamp(row.week_start if family == "descriptor" else row.start)
        days_before = 7 if family == "descriptor" else retained["controls"]["reference_days"]
        window = 7 if family == "descriptor" else retained["controls"]["window"]
        activity = "volume" if family == "descriptor" else retained["controls"]["activity_column"]
        days_after = 7 if family == "descriptor" else 0
        mask = (history.type_id == row.type_id) & history.market_date.between(start - pd.Timedelta(days=days_before), start + pd.Timedelta(days=window + days_after - 1))
        if family == "joint_paths":
            mask &= history.region_id == row.region_id
        h = history.loc[mask].sort_values("market_date")
        prior = h.loc[h.market_date < start]
        prior_price = pd.to_numeric(prior.vwap, errors="coerce").to_numpy(dtype=float)
        valid = prior.is_observed.fillna(False).to_numpy(dtype=bool) & np.isfinite(prior_price) & (prior_price > 0)
        prior_activity = pd.to_numeric(prior[activity], errors="coerce").to_numpy(dtype=float)
        if family == "descriptor":
            price_ref = np.median(prior_price[valid]) if valid.any() else np.nan
            activity_ref = prior_activity.mean() if len(prior_activity) and np.isfinite(prior_activity).all() and (prior_activity >= 0).all() else np.nan
        else:
            price_ref, activity_ref = row.price_reference, row.activity_reference
        raw_price = pd.to_numeric(h.vwap, errors="coerce").to_numpy(dtype=float)
        raw_price = np.where(h.is_observed.fillna(False).to_numpy(dtype=bool) & np.isfinite(raw_price) & (raw_price > 0), raw_price, np.nan)
        raw_activity = pd.to_numeric(h[activity], errors="coerce").to_numpy(dtype=float)
        price = raw_price / price_ref * 100 if np.isfinite(price_ref) and price_ref > 0 else np.full(len(h), np.nan)
        activity_ratio = raw_activity / activity_ref if np.isfinite(activity_ref) and activity_ref > 0 else np.full(len(h), np.nan)
        day = (h.market_date - start).dt.days
        for values, raw, channel, color, secondary in ((price, raw_price, "price index", "#1f77b4", False), (activity_ratio, raw_activity, f"{activity} ratio", "#ff7f0e", True)):
            fig.add_trace(go.Scatter(x=day, y=values, mode="lines", name=f"{label} · {channel}", line=dict(color=color, dash=style), connectgaps=False,
                customdata=raw, meta=str(batch.keys[i]), hovertemplate="day %{x}<br>relative %{y}<br>raw %{customdata}<br>%{meta}<extra>%{fullData.name}</extra>"), row=2, col=2, secondary_y=secondary)
    if family == "descriptor":
        fig.add_vrect(x0=-0.5, x1=6.5, fillcolor="gray", opacity=0.15, line_width=0,
                      annotation_text="focal week", row=2, col=2)
    fig.update_yaxes(title_text="Price index · reference 100", row=2, col=2, secondary_y=False)
    fig.update_yaxes(title_text="Activity / reference", row=2, col=2, secondary_y=True)
    geometry_label = "Aligned common-input geometry (not DTW)" if family == "joint_paths" else "Prepared descriptor common-input geometry"
    fig.update_layout(title=f"{geometry_label} · query {batch.keys[q]} · neighbor rank {neighbor_rank}: {batch.keys[neighbor]}", height=950)
    return fig

In [ ]:
plot_optional_input_geometry(*optional_input_args())

## 06 — Optional descriptor diagnostics

Choose one view below: temporal PCA, PCA variance/coefficients, or item influence. These views use the active descriptor execution's keyed inputs and retained PCA results. The default selects its highest-dimensional successful PCA representation; an explicit representation ID overrides that choice. A truncated PCA shows only its retained spectrum. No additional representation or clustering fits are performed.

Raw/prepared feature distributions and fitted-row correlations are available through the **Prepared / Raw** selector in the optional input board above.

In [ ]:
descriptor_diagnostic_view = "pca"  # temporal, pca, or item_influence
# A representation ID, not a clustering run ID. None uses the largest retained PCA.
diagnostic_pca_id = None
diagnostic_component_limit = 8
diagnostic_item_limit = 15

In [ ]:
def descriptor_diagnostic_context():
    """Read the active execution's aligned inputs and an already-fitted PCA."""
    if descriptor_diagnostic_view not in ("temporal", "pca", "item_influence"):
        raise ValueError("Choose temporal, pca, or item_influence")
    execution = require_active_execution()
    if execution.options["family"] != "descriptor":
        return {"reason": "Select a retained descriptor execution for these diagnostics."}
    batch = execution.batch
    candidates = [rep for rep in batch.representations.values()
                  if rep["status"] == "ok" and rep["params"]["method"] == "pca"]
    if diagnostic_pca_id is not None:
        rep = batch.representations.get(diagnostic_pca_id)
        if rep is None or rep["status"] != "ok" or rep["params"]["method"] != "pca":
            raise ValueError("diagnostic_pca_id must identify a successful PCA in the active execution")
    else:
        rep = min(candidates, key=lambda value: (-value["embedding"].shape[1],
                  value["params"].get("repetition", 0), value["id"])) if candidates else None
    if rep is not None and (tuple(rep["keys"]) != tuple(batch.keys)
                           or len(rep["embedding"]) != len(batch.keys)):
        raise ValueError("Retained PCA rows must match the active population")
    data = _inspection_data(execution)
    return {"reason": None, "batch": batch, "rows": data["rows"], "pca": rep,
            "label": f"descriptor population {batch.population_id[:12]} · {len(batch.keys):,} fitted windows"}


def descriptor_diagnostic_message(text):
    figure = go.Figure()
    figure.add_annotation(text=text, x=.5, y=.5, xref="paper", yref="paper", showarrow=False)
    figure.update_layout(xaxis_visible=False, yaxis_visible=False, height=300)
    return figure

### PCA variance and feature mixtures

The left panel shows individual and cumulative shares of total prepared-input variance, with eigenvalues in hover. The right panel shows component coefficients. A truncated fit need not reach 100%; explained variance alone does not establish useful behavioral groups.

In [ ]:
def plot_retained_pca_diagnostics(context, component_limit):
    if context["reason"]:
        return descriptor_diagnostic_message(context["reason"])
    rep = context["pca"]
    if rep is None:
        return descriptor_diagnostic_message("This execution has no successful PCA. Enable PCA in a future batch to inspect it.")
    if isinstance(component_limit, bool) or not isinstance(component_limit, int) or component_limit < 1:
        raise ValueError("component_limit must be a positive integer")
    model = rep["model"]
    count = min(component_limit, len(model.explained_variance_ratio_))
    components = np.arange(1, count + 1)
    ratios = model.explained_variance_ratio_[:count]
    names = list(context["batch"].features)
    coefficients = model.components_[:count].T
    figure = make_subplots(rows=1, cols=2, subplot_titles=(
        "Explained variance · retained components", "Component coefficients · prepared features"),
        horizontal_spacing=.18)
    figure.add_trace(go.Bar(x=components, y=ratios, name="Individual",
        customdata=model.explained_variance_[:count],
        hovertemplate="PC%{x}<br>variance share %{y:.2%}<br>eigenvalue %{customdata:.5g}<extra></extra>"), row=1, col=1)
    figure.add_trace(go.Scatter(x=components, y=np.cumsum(ratios), mode="lines+markers", name="Cumulative",
        hovertemplate="Through PC%{x}: %{y:.2%}<extra></extra>"), row=1, col=1)
    bound = float(np.max(np.abs(coefficients))) or 1.0
    figure.add_trace(go.Heatmap(x=[f"PC{i}" for i in components], y=names, z=coefficients,
        zmin=-bound, zmax=bound, zmid=0, colorscale="RdBu", colorbar_title="Coefficient",
        hovertemplate="%{y}<br>%{x}: %{z:.5g}<extra></extra>"), row=1, col=2)
    figure.update_xaxes(title_text="Component", dtick=1, row=1, col=1)
    figure.update_yaxes(title_text="Share of total prepared-input variance", tickformat=".0%", range=[0, 1], row=1, col=1)
    figure.update_layout(title=f"{context['label']}<br>PCA {rep['id'][:12]} · showing {count} of {len(model.explained_variance_ratio_)} fitted components",
        height=max(500, 20 * len(names) + 180), margin=dict(t=110), legend=dict(orientation="h"))
    return figure

In [ ]:
plot_retained_pca_diagnostics(descriptor_diagnostic_context(), diagnostic_component_limit) if descriptor_diagnostic_view == "pca" else None

### Which markets drive the geometry?

Compare each item's share of squared deviations from the population mean with its share of fitted observations. The dropdown selects total prepared-input variation or a retained PCA component. A large variation share can reflect displacement of an item's observations from the pooled mean as well as within-item variability; it is not an estimate of within-item volatility.

In [ ]:
def plot_retained_item_influence(context, item_limit):
    if context["reason"]:
        return descriptor_diagnostic_message(context["reason"])
    if isinstance(item_limit, bool) or not isinstance(item_limit, int) or item_limit < 1:
        raise ValueError("item_limit must be a positive integer")
    batch, rows, rep = context["batch"], context["rows"], context["pca"]
    centered = batch.prepared - batch.prepared.mean(axis=0)
    energies = {"Total prepared variation": np.square(centered).sum(axis=1)}
    if rep is not None:
        scores = rep["embedding"] - rep["embedding"].mean(axis=0)
        energies.update({f"PC{i + 1}": np.square(scores[:, i]) for i in range(scores.shape[1])})
    figure = go.Figure()
    metrics = []
    for metric, energy in energies.items():
        total = float(energy.sum())
        if not np.isfinite(total) or total <= 0:
            continue
        frame = rows.loc[:, ["type_id", "type_name"]].copy()
        frame["energy"] = energy
        ranked = frame.groupby(["type_id", "type_name"], as_index=False).agg(
            observations=("energy", "size"), energy=("energy", "sum"))
        ranked["variation_share"] = ranked.energy / total
        ranked["observation_share"] = ranked.observations / len(rows)
        ranked = ranked.sort_values(["variation_share", "type_id"], ascending=[False, True]).head(item_limit)
        labels = [f"{r.type_name} ({r.type_id})" for r in ranked.itertuples()]
        hover = np.column_stack((ranked.observations, ranked.observation_share, ranked.variation_share))
        for label, values, color in (("Variation share", ranked.variation_share, "#0072B2"),
                                      ("Observation share", ranked.observation_share, "#E69F00")):
            figure.add_trace(go.Bar(x=values, y=labels, orientation="h", name=label,
                marker_color=color, visible=not metrics, customdata=hover,
                hovertemplate="%{y}<br>variation share %{customdata[2]:.2%}<br>observation share %{customdata[1]:.2%}<br>fitted windows %{customdata[0]:,.0f}<extra></extra>"))
        metrics.append(metric)
    if not metrics:
        return descriptor_diagnostic_message("No positive finite variation is available.")
    buttons = []
    for index, metric in enumerate(metrics):
        visible = [False] * len(figure.data)
        visible[2 * index:2 * index + 2] = [True, True]
        buttons.append(dict(label=metric, method="update", args=[{"visible": visible},
            {"title.text": f"{context['label']}<br>{metric} · top {item_limit} items"}]))
    figure.update_layout(title=f"{context['label']}<br>{metrics[0]} · top {item_limit} items",
        barmode="group", height=650, margin=dict(l=230, t=150),
        xaxis_title="Share of fitted observations or variation", yaxis_title="Item",
        updatemenus=[dict(buttons=buttons, active=0, x=0, y=1.13, xanchor="left")])
    figure.update_xaxes(tickformat=".0%")
    figure.update_yaxes(autorange="reversed")
    return figure

In [ ]:
plot_retained_item_influence(descriptor_diagnostic_context(), diagnostic_item_limit) if descriptor_diagnostic_view == "item_influence" else None

### Temporal PCA behavior explorer

The existing offline explorer shows the selected retained PCA's first three coordinates, with item/category and calendar controls, trajectories, and full/central ranges. Dates mark focal-week ends. Missing eligible weeks break trajectories. This is a view of the descriptor population and its retrospective context; it does not refit PCA or change clustering assignments.

In [ ]:
def retained_temporal_pca_model(context):
    """Adapt keyed retained scores to the existing temporal viewer's input."""
    from market_analysis.plotting.temporal_pca import prepare_temporal_pca
    rows, rep = context["rows"], context["pca"]
    coordinates = np.asarray(rep["embedding"][:, :3], dtype=float)
    ids, inverse = np.unique(rows.type_id.to_numpy(), return_inverse=True)
    group_rows = [np.flatnonzero(inverse == i) for i in range(len(ids))]
    categories = rows.study_category.astype(str).to_numpy()
    names = sorted(set(categories))
    palette = {name: f"hsl({i * 360 / len(names):.1f}, 65%, 42%)" for i, name in enumerate(names)}
    return prepare_temporal_pca({
        "coordinates": coordinates,
        "dates": pd.to_datetime(rows.week_end).to_numpy(dtype="datetime64[D]"),
        "week_starts": pd.to_datetime(rows.week_start).to_numpy(dtype="datetime64[D]"),
        "study_categories": categories, "category_colors": palette, "expected_step_days": 7,
        "type_ids": ids, "labels": [str(rows.iloc[index[0]].type_name) for index in group_rows],
        "group_rows": group_rows,
        "central_ranges": [np.quantile(coordinates[:, i], [.01, .99]).tolist() for i in range(3)],
    })


def plot_retained_temporal_pca(model, context):
    from market_analysis.plotting.temporal_pca import build_temporal_pca_figure
    rep = context["pca"]
    figure = build_temporal_pca_figure(model, title=f"Temporal PCA · {context['label']} · PCA {rep['id'][:12]}")
    for i, axis in enumerate(("xaxis", "yaxis", "zaxis")):
        figure.layout.scene[axis].title.text = f"PC{i + 1} ({rep['model'].explained_variance_ratio_[i]:.1%})"
    return figure


def render_retained_temporal_pca(context):
    from IPython.display import HTML
    from market_analysis.plotting.temporal_pca import temporal_pca_html
    if context["reason"]:
        return descriptor_diagnostic_message(context["reason"])
    if context["pca"] is None or context["pca"]["embedding"].shape[1] < 3:
        return descriptor_diagnostic_message("Temporal PCA requires a successful retained PCA with at least three components.")
    model = retained_temporal_pca_model(context)
    return HTML(temporal_pca_html(plot_retained_temporal_pca(model, context), model))

In [ ]:
render_retained_temporal_pca(descriptor_diagnostic_context()) if descriptor_diagnostic_view == "temporal" else None